In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')

In [ ]:
df.columns

In [ ]:
df.head()

# FINALE

In [ ]:
import pandas as pd
import numpy as np

# Tri par période pour les features chronologiques
df = df.sort_values(by='period').reset_index(drop=True)

print("DataFrame sorted by 'period'.")

# --- Features générales, toutes transactions ---

# 1. Comptage cumulé des transactions par type d'opération et par compte
df['origin_operation_cumcount'] = df.groupby(['origin_account', 'operation']).cumcount() + 1
df['destination_operation_cumcount'] = df.groupby(['destination_account', 'operation']).cumcount() + 1

# 2. Comptage cumulé total des transactions par compte
df['origin_total_cumcount'] = df.groupby('origin_account').cumcount() + 1
df['destination_total_cumcount'] = df.groupby('destination_account').cumcount() + 1

# 3. Solde après la transaction précédente (décalé) et écart
df['origin_prev_balance_after'] = df.groupby('origin_account')['origin_balance_after'].shift(1).fillna(df['origin_balance_before'])
df['destination_prev_balance_after'] = df.groupby('destination_account')['destination_balance_after'].shift(1).fillna(df['destination_balance_before'])

df['origin_balance_before_vs_prev_after_diff'] = df['origin_balance_before'] - df['origin_prev_balance_after']
df['destination_balance_before_vs_prev_after_diff'] = df['destination_balance_before'] - df['destination_prev_balance_after']

# 4. Délai depuis la dernière opération (global et par type)
df['origin_time_since_last_txn'] = df.groupby('origin_account')['period'].diff().fillna(0)
df['destination_time_since_last_txn'] = df.groupby('destination_account')['period'].diff().fillna(0)

df['origin_time_since_last_op_type'] = df.groupby(['origin_account', 'operation'])['period'].diff().fillna(0)
df['destination_time_since_last_op_type'] = df.groupby(['destination_account', 'operation'])['period'].diff().fillna(0)

print("General features engineered (cumulative counts, shifted balances, time differences).")

# --- Features : loi géométrique, op_03 uniquement ---

# Métriques de fraude pour un type de compte
def get_fraud_time_metrics(account_col, df_data):
    # Uniquement les comptes ayant fraudé au moins une fois
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_col].unique()

    # Période de première apparition et de première fraude
    first_appearance = df_data.groupby(account_col)['period'].min()
    first_fraud = df_data[df_data['fraud_flag'] == 1].groupby(account_col)['period'].min()

    # Délai jusqu'à la première fraude
    time_to_first_fraud = (first_fraud - first_appearance).fillna(-1) # -1 pour les comptes sans fraude

    # Nombre de transactions avant la première fraude
    num_txns_until_first_fraud = {}
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_col] == account]

        # Période de la première fraude du compte
        if account in first_fraud.index:
            f_fraud_period = first_fraud.loc[account]

            # Transactions de la première apparition jusqu'à la première fraude incluse
            txns_slice = account_txns[account_txns['period'] <= f_fraud_period]
            num_txns_until_first_fraud[account] = len(txns_slice)
        else:
            num_txns_until_first_fraud[account] = -1

    time_to_first_fraud_series = time_to_first_fraud[fraudulent_accounts] # comptes frauduleux uniquement
    num_txns_until_first_fraud_series = pd.Series(num_txns_until_first_fraud, index=fraudulent_accounts)

    return time_to_first_fraud_series, num_txns_until_first_fraud_series

# Métriques de fraude côté émetteur et côté destinataire
origin_time_to_first_fraud, origin_num_txns_to_first_fraud = get_fraud_time_metrics('origin_account', df)
destination_time_to_first_fraud, destination_num_txns_to_first_fraud = get_fraud_time_metrics('destination_account', df)

# Fusion dans le DataFrame principal
df['origin_time_to_first_fraud'] = df['origin_account'].map(origin_time_to_first_fraud).fillna(-1)
df['origin_num_txns_to_first_fraud'] = df['origin_account'].map(origin_num_txns_to_first_fraud).fillna(-1)
df['destination_time_to_first_fraud'] = df['destination_account'].map(destination_time_to_first_fraud).fillna(-1)
df['destination_num_txns_to_first_fraud'] = df['destination_account'].map(destination_num_txns_to_first_fraud).fillna(-1)

# Probabilités géométriques, op_03 uniquement
def calculate_p_geometric(value):
    return 1 / (value + 1) if value >= 0 else -1 # gestion de la valeur -1

df['p_geometric_origin_period'] = df['origin_time_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_origin_txns'] = df['origin_num_txns_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_destination_period'] = df['destination_time_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_destination_txns'] = df['destination_num_txns_to_first_fraud'].apply(calculate_p_geometric)

# Features géométriques sur op_03, -1 ailleurs
for col in ['p_geometric_origin_period', 'p_geometric_origin_txns',
            'p_geometric_destination_period', 'p_geometric_destination_txns']:
    df.loc[df['operation'] != 'op_03', col] = -1

print("Geometric probability features engineered for 'op_03' transactions.")

print("New features have been added to the DataFrame.")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

print("--- Building Fraud Prediction Model for 'op_03' Transactions ---")

# Restriction aux opérations op_03
df_op03 = df[df['operation'] == 'op_03'].copy()

# Tri par période pour les features chronologiques
df_op03 = df_op03.sort_values(by='period').reset_index(drop=True)
print(f"DataFrame filtered for 'op_03' transactions. Shape: {df_op03.shape}")

# --- Features recalculées dans le contexte op_03 ---

# 1. Features séquentielles appliquées à df_op03
df_op03['origin_transaction_sequence'] = df_op03.groupby('origin_account').cumcount() + 1
df_op03['destination_transaction_sequence'] = df_op03.groupby('destination_account').cumcount() + 1
df_op03['origin_dest_pair_sequence'] = df_op03.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2. Drapeaux d'historique de fraude
# Ces drapeaux reposent sur l'historique complet : on les redérive depuis df
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

df_op03['origin_account_previously_fraud'] = df_op03['origin_account'].isin(all_fraudulent_accounts).astype(int)
df_op03['destination_account_previously_fraud'] = df_op03['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("Essential feature engineering completed for df_op03.")

# Cible
y = df_op03['fraud_flag']

# Features du modèle
# Hors id, operation, origin_account, destination_account et la cible fraud_flag
features_to_exclude = [
    'id', 'operation', 'origin_account', 'destination_account', 'fraud_flag'
]

# Features candidates présentes dans df_op03
all_possible_features = [
    'period', 'amount',
    'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_operation_cumcount', 'destination_operation_cumcount',
    'origin_total_cumcount', 'destination_total_cumcount',
    'origin_prev_balance_after', 'destination_prev_balance_after',
    'origin_balance_before_vs_prev_after_diff', 'destination_balance_before_vs_prev_after_diff',
    'origin_time_since_last_txn', 'destination_time_since_last_txn',
    'origin_time_since_last_op_type', 'destination_time_since_last_op_type',
    'origin_time_to_first_fraud', 'origin_num_txns_to_first_fraud',
    'destination_time_to_first_fraud', 'destination_num_txns_to_first_fraud',
    'p_geometric_origin_period', 'p_geometric_origin_txns',
    'p_geometric_destination_period', 'p_geometric_destination_txns',
    'origin_transaction_sequence', 'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'origin_account_previously_fraud', 'destination_account_previously_fraud'
]

# On ne garde que les features réellement présentes et non exclues
features = [f for f in all_possible_features if f in df_op03.columns and f not in features_to_exclude]
X = df_op03[features]

# NaN résiduels remplis à 0
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost
    model_op03_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False
    )
    model_op03_fraud.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_op03_fraud.predict(X_test)
    y_pred_proba = model_op03_fraud.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for 'op_03' Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title("Confusion Matrix for 'op_03' Fraud Prediction")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    globals()['model_op03_fraud'] = model_op03_fraud
    globals()['X_train_op03'] = X_train
    globals()['y_train_op03'] = y_train
    globals()['X_test_op03'] = X_test
    globals()['y_test_op03'] = y_test
    globals()['y_pred_proba_op03'] = y_pred_proba

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

print("--- Building Fraud Prediction Model for 'op_03' Transactions (using existing features) ---")

# Restriction aux opérations op_03
df_op03 = df[df['operation'] == 'op_03'].copy()

# Tri par période
df_op03 = df_op03.sort_values(by='period').reset_index(drop=True)
print(f"DataFrame filtered for 'op_03' transactions. Shape: {df_op03.shape}")

# Cible
y = df_op03['fraud_flag']

# Features du modèle
# Hors id, operation, origin_account, destination_account et la cible fraud_flag
features_to_exclude = [
    'id', 'operation', 'origin_account', 'destination_account', 'fraud_flag'
]

# Features candidates présentes dans df_op03
all_possible_features = [
    'period', 'amount',
    'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_operation_cumcount', 'destination_operation_cumcount',
    'origin_total_cumcount', 'destination_total_cumcount',
    'origin_prev_balance_after', 'destination_prev_balance_after',
    'origin_balance_before_vs_prev_after_diff', 'destination_balance_before_vs_prev_after_diff',
    'origin_time_since_last_txn', 'destination_time_since_last_txn',
    'origin_time_since_last_op_type', 'destination_time_since_last_op_type',
    #'origin_time_to_first_fraud', 'origin_num_txns_to_first_fraud',
    #'destination_time_to_first_fraud', 'destination_num_txns_to_first_fraud',
    'p_geometric_origin_period', 'p_geometric_origin_txns',
    'p_geometric_destination_period', 'p_geometric_destination_txns',
    'origin_transaction_sequence', 'destination_transaction_sequence',
    'origin_dest_pair_sequence'
    #'origin_account_previously_fraud', 'destination_account_previously_fraud'
]

# On ne garde que les features réellement présentes et non exclues
features = [f for f in all_possible_features if f in df_op03.columns and f not in features_to_exclude]
X = df_op03[features]

# NaN résiduels remplis à 0
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost
    model_op03_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False
    )
    model_op03_fraud.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_op03_fraud.predict(X_test)
    y_pred_proba = model_op03_fraud.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for 'op_03' Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title("Confusion Matrix for 'op_03' Fraud Prediction")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    globals()['model_op03_fraud'] = model_op03_fraud
    globals()['X_train_op03'] = X_train
    globals()['y_train_op03'] = y_train
    globals()['X_test_op03'] = X_test
    globals()['y_test_op03'] = y_test
    globals()['y_pred_proba_op03'] = y_pred_proba

# old

In [ ]:
df_sorted = df.sort_values(by='period').reset_index(drop=True)

# 1. Rang de la transaction pour chaque compte émetteur
df_sorted['origin_transaction_sequence'] = df_sorted.groupby('origin_account').cumcount() + 1

# 2. Rang de la transaction pour chaque compte destinataire
df_sorted['destination_transaction_sequence'] = df_sorted.groupby('destination_account').cumcount() + 1

# 3. Rang de la rencontre pour un couple émetteur-destinataire
df_sorted['origin_dest_pair_sequence'] = df_sorted.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Fusion des nouvelles features dans df

df = df_sorted

print("New sequential transaction features created:")
print(df[['id', 'period', 'origin_account', 'origin_transaction_sequence',
          'destination_account', 'destination_transaction_sequence',
          'origin_dest_pair_sequence']].head())


In [ ]:
import numpy as np

# Comptes émetteurs impliqués dans une fraude
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()

# Comptes destinataires impliqués dans une fraude
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Ensemble des comptes ayant fraudé au moins une fois (émetteur ou destinataire)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Drapeaux d'historique de fraude pour l'émetteur et le destinataire
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("New columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

In [ ]:
df_op03 = df[df['operation'] == 'op_03'].copy()

unique_op03_dest_accounts = df_op03['destination_account'].unique()
num_unique_op03_dest_accounts = len(unique_op03_dest_accounts)

print(f"Nombre de comptes destinataires uniques dans les opérations de type 'op_03': {num_unique_op03_dest_accounts}")

# fraudulent_destination_accounts : tous les comptes destinataires ayant déjà fraudé

# Comptes destinataires op_03 absents de fraudulent_destination_accounts
non_fraudulent_op03_dest_accounts = [account for account in unique_op03_dest_accounts if account not in fraudulent_destination_accounts]
num_non_fraudulent_op03_dest_accounts = len(non_fraudulent_op03_dest_accounts)

print(f"Parmi eux, le nombre de comptes destinataires qui n'ont jamais fraudé est: {num_non_fraudulent_op03_dest_accounts}")


In [ ]:
import pandas as pd


# 1. Transactions des comptes destinataires jamais frauduleux en op_03
transactions_non_fraud_op03_dest = df_op03[
df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)
]
num_transactions_non_fraud_op03_dest = len(transactions_non_fraud_op03_dest)

print(f"Nombre de transactions 'op_03' impliquant les 4593 comptes destinataires n'ayant jamais été frauduleux : {num_transactions_non_fraud_op03_dest}")

# 2. Transactions des comptes destinataires ayant fraudé au moins une fois
transactions_fraud_op03_dest = df_op03[
df_op03['destination_account'].isin(fraudulent_destination_accounts)
]
num_transactions_fraud_op03_dest = len(transactions_fraud_op03_dest)

print(f"Nombre de transactions 'op_03' impliquant les comptes destinataires ayant été frauduleux au moins une fois : {num_transactions_fraud_op03_dest}")

# 3. Distribution de fraud_flag dans chaque groupe
print("\nRépartition de 'fraud_flag' pour les transactions des comptes destinataires non-frauduleux (théoriquement tous 0) :")
print(transactions_non_fraud_op03_dest['fraud_flag'].value_counts(normalize=True))

print("\nRépartition de 'fraud_flag' pour les transactions des comptes destinataires frauduleux au moins une fois :")
print(transactions_fraud_op03_dest['fraud_flag'].value_counts(normalize=True))

In [ ]:
import pandas as pd


# Transactions des comptes destinataires ayant fraudé au moins une fois
df_fraud_dest_accounts_only = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Tri par période pour repérer la première apparition
df_fraud_dest_accounts_only = df_fraud_dest_accounts_only.sort_values(by=['destination_account', 'period'])

# Première transaction de chacun de ces comptes destinataires
first_appearance_transactions = df_fraud_dest_accounts_only.groupby('destination_account').first().reset_index()

print("Première apparition des comptes destinataires ayant été frauduleux (avec leur fraud_flag à ce moment):")
print(first_appearance_transactions[['destination_account', 'period', 'fraud_flag']].head())

print(
    "\nDistribution du fraud_flag pour la première apparition de ces comptes destinataires (tous devraient être 0 si le premier signal est toujours non-frauduleux, mais un 1 indiquerait une fraude immédiate):"
)
print(first_appearance_transactions['fraud_flag'].value_counts(normalize=True))
print(first_appearance_transactions['fraud_flag'].value_counts())

# Y a-t-il des fraud_flag = 1 dès la première apparition ?
if (first_appearance_transactions['fraud_flag'] == 1).any():
    print(
        "\nComptes destinataires dont la première transaction enregistrée est déjà frauduleuse:"
    )
    print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 1][['destination_account', 'period', 'fraud_flag']])
else:
    print(
        "\nAucun compte destinataire n'a été enregistré comme frauduleux dès sa première apparition."
    )


In [ ]:
import pandas as pd


# 1. Comptes destinataires non frauduleux (flag=0) à leur première apparition
#    mais frauduleux (flag=1) au moins une fois par la suite
# On filtre sur fraud_flag == 0 à la première apparition

initial_non_fraudulent_then_fraud_accounts = first_appearance_transactions[
    first_appearance_transactions['fraud_flag'] == 0
]['destination_account'].unique()

print(f"Number of destination accounts that were initially non-fraudulent then became fraudulent: {len(initial_non_fraudulent_then_fraud_accounts)}")

# 2. Toutes les transactions de ces comptes
df_dest_nonfraud_then_fraud = df[
    df['destination_account'].isin(initial_non_fraudulent_then_fraud_accounts)
].copy()

print(f"Created `df_dest_nonfraud_then_fraud` with shape: {df_dest_nonfraud_then_fraud.shape}")

print("First 5 rows of the new dataset:")
print(df_dest_nonfraud_then_fraud.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Transactions dont le compte destinataire a déjà fraudé
df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Analyse détaillée par compte
account_timeline_data = []

# Parcours des comptes destinataires frauduleux
for account in fraudulent_destination_accounts:
    # Transactions du compte courant, triées par période
    account_df = df_fraud_dest_txns[df_fraud_dest_txns['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    # Initialisation du premier segment
    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    # Détection des changements de statut
    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            # Fin du segment précédent
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1 # période incluse

            account_timeline_data.append({
                'account': account,
                'status': current_status, # 0 = non-fraude, 1 = fraude
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            # Nouveau segment
            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    # Dernier segment après la boucle
    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

# Passage en DataFrame
timeline_df = pd.DataFrame(account_timeline_data)

print("\nAnalyse de la chronologie des statuts de fraude par compte destinataire:")
display(timeline_df.head())

# Synthèse des segments frauduleux (statut=1) et non frauduleux (statut=0)
summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

print("\nStatistiques récapitulatives des phases (frauduleuses vs. non-frauduleuses):")
display(summary_stats)

# Distributions de la durée et du nombre de transactions par segment
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
sns.histplot(timeline_df[timeline_df['status'] == 0]['duration'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['duration'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution des Durées de Phase par Statut de Fraude')
plt.xlabel('Durée (Périodes)')
plt.ylabel('Fréquence')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(timeline_df[timeline_df['status'] == 0]['num_transactions'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['num_transactions'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution du Nombre de Transactions par Statut de Fraude')
plt.xlabel('Nombre de Transactions')
plt.ylabel('Fréquence')
plt.legend()

plt.tight_layout()
plt.show()

# Nombre de changements de statut par compte
# Un changement = fraud_flag qui change d'une transaction à la suivante
# Sur le df complet trié par période, groupé par compte

# Colonne intermédiaire : variation de fraud_flag dans la chronologie du compte
relevant_transactions_sorted = df_fraud_dest_txns.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

# Comptage des changements réels (0 vers 1 ou 1 vers 0)
# Un diff non nul indique un changement de statut
num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

print("\nDistribution du nombre de changements de statut (Non-Fraude <-> Fraude) par compte:")
# Comptes sans changement (un seul segment) : 0 changement
all_accounts_in_relevant_txns = df_fraud_dest_txns['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)

display(num_status_changes_per_account_full.describe())

# Distribution du nombre de changements de statut
plt.figure(figsize=(10, 6))
# Bins de 0 à max_changes + 1 pour un comptage par valeur entière
bins = np.arange(num_status_changes_per_account_full.max() + 2) - 0.5
sns.histplot(num_status_changes_per_account_full, bins=bins, kde=False)
plt.title('Nombre de Changements de Statut par Compte Destinataire Frauduleux')
plt.xlabel('Nombre de Changements de Statut')
plt.ylabel('Nombre de Comptes')
plt.xticks(np.arange(0, num_status_changes_per_account_full.max() + 1, 1))
plt.show()


#### model qui predit le premier flaque d'un compte untilisateur sa nature

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Training XGBoost model on first appearance of eventually fraudulent destination accounts ---")

# Copie de first_appearance_transactions pour ce modèle
df_first_appearance_model = first_appearance_transactions.copy()

# Tri par période pour des cumuls cohérents
df_first_appearance_model = df_first_appearance_model.sort_values(by='period').reset_index(drop=True)

# --- Nouvelles features ---

# Cumul de valeurs uniques sur une série non numérique
def cumulative_nunique(series):
    unique_elements = set()
    counts = []
    for x in series:
        unique_elements.add(x)
        counts.append(len(unique_elements))
    return pd.Series(counts, index=series.index)

# 1. Nombre de transferts par type d'opération et par émetteur (cumulé)
# Pour chaque émetteur et chaque type d'opération, nombre d'occurrences
df_first_appearance_model['origin_op_cumulative_count'] = df_first_appearance_model.groupby(['origin_account', 'operation']).cumcount() + 1

# 2. Nombre de comptes distincts rencontrés (cumulé)
# Nombre de destinataires distincts par émetteur
df_first_appearance_model['origin_unique_dest_cumulative_count'] = (
    df_first_appearance_model.groupby('origin_account')['destination_account']
    .transform(cumulative_nunique)
)
# Nombre d'émetteurs distincts par destinataire
df_first_appearance_model['dest_unique_origin_cumulative_count'] = (
    df_first_appearance_model.groupby('destination_account')['origin_account']
    .transform(cumulative_nunique)
)

# 3. Délai depuis la transaction précédente du compte
# Délai depuis la dernière transaction du compte émetteur
df_first_appearance_model['origin_time_since_last_txn'] = df_first_appearance_model.groupby('origin_account')['period'].diff().fillna(0)
# Délai depuis la dernière transaction du compte destinataire
# Vaut 0 ici : chaque compte destinataire n'apparaît qu'une fois à sa première apparition
df_first_appearance_model['dest_time_since_last_txn'] = df_first_appearance_model.groupby('destination_account')['period'].diff().fillna(0)


# 4. Délai depuis la dernière transaction du même type pour le destinataire
# Délai depuis la dernière opération de même type pour le destinataire
# Vaut 0 ici : chaque compte destinataire n'apparaît qu'une fois à sa première apparition
df_first_appearance_model['dest_op_time_since_last_txn'] = df_first_appearance_model.groupby(['destination_account', 'operation'])['period'].diff().fillna(0)

print("New features 'origin_op_cumulative_count', 'origin_unique_dest_cumulative_count', 'dest_unique_origin_cumulative_count', 'origin_time_since_last_txn', 'dest_time_since_last_txn', and 'dest_op_time_since_last_txn' created.")

# --- Fin des nouvelles features ---

# One-hot encoding de operation
df_first_appearance_model = pd.get_dummies(df_first_appearance_model, columns=['operation'], prefix='operation', drop_first=True)

# Cible y
y = df_first_appearance_model['fraud_flag']

# Features X
# Hors id, destination_account, origin_account et la cible fraud_flag
# Les features séquentielles constantes sont écartées
constant_seq_cols = []
for col in ['origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence']:
    if col in df_first_appearance_model.columns and df_first_appearance_model[col].nunique() == 1:
        constant_seq_cols.append(col)

# origin_account et destination_account exclus (type object)

for new_feature_col in ['dest_unique_origin_cumulative_count', 'dest_time_since_last_txn', 'dest_op_time_since_last_txn']:
    if new_feature_col in df_first_appearance_model.columns and df_first_appearance_model[new_feature_col].nunique() == 1:
        constant_seq_cols.append(new_feature_col)

features_to_exclude = ['id', 'destination_account', 'origin_account', 'fraud_flag'] + constant_seq_cols
X = df_first_appearance_model.drop(columns=features_to_exclude, errors='ignore')

# NaN résiduels remplis à 0
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost
    model_first_appearance_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False
    )
    model_first_appearance_fraud.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_first_appearance_fraud.predict(X_test)
    y_pred_proba = model_first_appearance_fraud.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for First Appearance Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'] ,
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for First Appearance Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if 'X' in globals() and 'y' in globals():
    # Features et cible réunies pour la corrélation
    df_for_correlation = pd.concat([X, y], axis=1)

    # Matrice de corrélation
    correlation_matrix = df_for_correlation.corr()

    # Corrélations avec la cible fraud_flag
    fraud_correlation = correlation_matrix['fraud_flag'].sort_values(ascending=False)

    print("Correlation of features with 'fraud_flag' (descending order):\n")
    print(fraud_correlation)

    # Heatmap des corrélations
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix[['fraud_flag']].sort_values(by='fraud_flag', ascending=False), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Feature Correlation with Fraud Flag')
    plt.show()

else:
    print("Erreur : X ou y introuvable. Exécuter d'abord la cellule d'entraînement du modèle.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


print("Descriptive statistics for 'period' at first appearance for fraud_flag = 0:")
print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 0]['period'].describe())

print("\nDescriptive statistics for 'period' at first appearance for fraud_flag = 1:")
print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 1]['period'].describe())

# Distribution de period par classe de fraud_flag
plt.figure(figsize=(10, 6))
sns.boxplot(x='fraud_flag', y='period', data=first_appearance_transactions)
plt.title('Distribution of Period at First Appearance by Fraud Flag')
plt.xlabel('Fraud Flag at First Appearance (0 = Non-Fraudulent, 1 = Fraudulent)')
plt.ylabel('Period')
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=first_appearance_transactions, x='period', hue='fraud_flag', fill=True, common_norm=False)
plt.title('KDE Plot of Period at First Appearance by Fraud Flag')
plt.xlabel('Period')
plt.ylabel('Density')
plt.grid(True)
plt.show()

print("\nInterpretation:\n- By comparing the descriptive statistics and the box/KDE plots, we can observe if accounts that are initially flagged as fraudulent (fraud_flag=1) tend to appear earlier or later in the dataset's timeline ('period') compared to accounts that are initially non-fraudulent (fraud_flag=0) but later become fraudulent.")

### suite

In [ ]:
import pandas as pd

# Transactions op_03 vers des comptes destinataires non frauduleux
transactions_non_fraudulent_dest_op03 = df_op03[df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)]

# Périodes distinctes de ces transactions
unique_periods_non_fraud_dest_op03 = sorted(transactions_non_fraudulent_dest_op03['period'].unique())

print("Periods in which 'op_03' transactions occur with never fraudulent destination accounts:")
print(unique_periods_non_fraud_dest_op03)

In [ ]:
import pandas as pd

if 'df_op03' not in locals():
    df_op03 = df[df['operation'] == 'op_03'].copy()

# 1. Transactions op_03 jamais frauduleuses
df_never_fraudulent_op03 = df_op03[df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)].copy()

# 2. Transactions op_03 marquées comme frauduleuses
df_only_fraudulent_op03_txns = df_op03[df_op03['fraud_flag'] == 1].copy()

print("DataFrame 'df_never_fraudulent_op03' created (transactions with never-fraudulent destination accounts in op_03).")
print(f"Shape of df_never_fraudulent_op03: {df_never_fraudulent_op03.shape}")
print("First 5 rows of df_never_fraudulent_op03:")
print(df_never_fraudulent_op03.head())

print("\nDataFrame 'df_only_fraudulent_op03_txns' created (transactions with fraud_flag = 1 in op_03).")
print(f"Shape of df_only_fraudulent_op03_txns: {df_only_fraudulent_op03_txns.shape}")
print("First 5 rows of df_only_fraudulent_op03_txns:")
print(df_only_fraudulent_op03_txns.head())

In [ ]:
df_never_fraudulent_op03.tail()

In [ ]:
df_only_fraudulent_op03_txns.tail()

### petit detour

In [ ]:
import pandas as pd
import numpy as np

# --- destination_fraud_metrics ---
# Tri par période : indispensable pour la première apparition et la première fraude
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

# Seuls les comptes destinataires servent à l'étape suivante
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)


# Moyenne de time_to_fraud pour les comptes destinataires
mean_time_to_fraud_dest = destination_fraud_metrics['time_to_fraud'].mean()

p_geometric = 1 / (mean_time_to_fraud_dest + 1)

print(f"The calculated parameter 'p' for the geometric distribution of 'time_to_fraud' in destination accounts is: {p_geometric:.4f}")

# --- Stratégie de censure ---
# On ne garde comme non-fraudes que les comptes restés sains longtemps

# 1. Comptes distincts du jeu de données
all_unique_accounts = pd.concat([df['origin_account'], df['destination_account']]).unique()

# 2. Comptes ayant déjà fraudé, en émetteur ou en destinataire
all_fraudulent_accounts_set = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# 3. Comptes jamais frauduleux : présents dans all_unique_accounts, absents de all_fraudulent_accounts
truly_non_fraudulent_accounts_set = set(all_unique_accounts) - all_fraudulent_accounts_set

# 4. Durée de vie de ces comptes (max_period - min_period)
#    Première et dernière période d'apparition de chaque compte
first_appearance_all = df.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})

account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']

# 5. Restriction de account_lifetimes aux comptes destinataires jamais frauduleux
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_set)].copy()

# 6. Seuil de stabilité, dérivé de la moyenne de la loi géométrique
#    Moyenne à 2.19 périodes, seuil fixé à 3 fois cette moyenne
long_time_threshold_periods = int(mean_time_to_fraud_dest * 3) # conversion en entier
if long_time_threshold_periods == 0: long_time_threshold_periods = 1 # au moins une période
print(f"\nDefining 'long time' for stable accounts as >= {long_time_threshold_periods} periods of activity without fraud.")

# 7. Comptes destinataires jamais frauduleux et stables sur la durée
stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[
    truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods
]['destination_account'].tolist())

# 8. Application de la censure dans df_filtered_for_training
#    Toutes les transactions frauduleuses (fraud_flag == 1) sont conservées
#    Les non-fraudes (fraud_flag == 0) ne le sont que si leur compte destinataire
#    appartient à stable_long_time_accounts_set

df_filtered_for_training = df[
    (df['fraud_flag'] == 1) | # on garde toutes les fraudes
    ((df['fraud_flag'] == 0) & (df['destination_account'].isin(stable_long_time_accounts_set))) # non-fraudes des comptes stables uniquement
].copy()

print(f"\nOriginal DataFrame shape: {df.shape}")
print(f"Shape of filtered DataFrame for training (after censoring recent healthy accounts): {df_filtered_for_training.shape}")
print(f"Number of transactions censored: {df.shape[0] - df_filtered_for_training.shape[0]}")

# Aperçu du DataFrame filtré
print("\nFirst 5 rows of df_filtered_for_training:")
print(df_filtered_for_training.head())

In [ ]:
df_filtered_for_training['operation'].value_counts()


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

# Copie de travail
df_train = df_filtered_for_training.copy()

# --- Fonctions utilitaires ---
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            # Garde-fou contre les dates invalides, trop anciennes ou futures
            if 0 < ts_sec < 4102444800: # du 1er janvier 1970 au 1er janvier 2100
                return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
            else:
                return None
    except (ValueError, TypeError, OSError):
        pass
    return pd.NaT

def extract_decimal_features(value):
    if pd.isna(value):
        return False, 0, 0
    s = str(value)
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}"
        s = s.rstrip('0').rstrip('.')
    has_decimal = '.' in s
    if has_decimal:
        parts = s.split('.')
        integer_part = parts[0]
        decimal_part = parts[1]
        num_decimal_places = len(decimal_part)
    else:
        integer_part = s
        num_decimal_places = 0
    num_digits_before_decimal = len(integer_part.lstrip('-'))
    return has_decimal, num_decimal_places, num_digits_before_decimal

def get_first_four_digits(value):
    if pd.isna(value):
        return [np.nan] * 4
    s = str(value)
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}"
    s_digits = s.replace('.', '').lstrip('-')
    digits = []
    for i in range(4):
        if i < len(s_digits):
            digits.append(int(s_digits[i]))
        else:
            digits.append(np.nan)
    return digits

# --- Classement des comptes ---
df_train['origin_account_stripped'] = df_train['origin_account'].str.replace('acc_o_', '')
df_train['destination_account_stripped'] = df_train['destination_account'].str.replace('acc_d_', '')

all_stripped_accounts = pd.concat([
    df_train['origin_account_stripped'],
    df_train['destination_account_stripped']
]).unique()

sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

# --- Features hexadécimales sur df_train ---
df_train['origin_hex_clean'] = df_train['origin_account'].apply(clean_hex)
df_train['destination_hex_clean'] = df_train['destination_account'].apply(clean_hex)

df_train['calc_origin_decimal']   = df_train['origin_hex_clean'].apply(hex_to_decimal)
df_train['calc_origin_ipv4']      = df_train['origin_hex_clean'].apply(hex_to_ipv4)
df_train['calc_origin_mac']       = df_train['origin_hex_clean'].apply(hex_to_mac)
df_train['calc_origin_timestamp'] = df_train['origin_hex_clean'].apply(hex_to_timestamp)

df_train['calc_destination_decimal']   = df_train['destination_hex_clean'].apply(hex_to_decimal)
df_train['calc_destination_ipv4']      = df_train['destination_hex_clean'].apply(hex_to_ipv4)
df_train['calc_destination_mac']       = df_train['destination_hex_clean'].apply(hex_to_mac)
df_train['calc_destination_timestamp'] = df_train['destination_hex_clean'].apply(hex_to_timestamp)

df_train.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')

df_train['has_origin_timestamp'] = df_train['calc_origin_timestamp'].notna()
df_train['has_destination_timestamp'] = df_train['calc_destination_timestamp'].notna()

# --- Classement des comptes sur df_train ---
# Réutilisation de account_to_rank_mapping
df_train['origin_account_ranked'] = df_train['origin_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)
df_train['destination_account_ranked'] = df_train['destination_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)

# --- Features de décimales et de chiffres sur df_train ---
numerical_cols = ['amount', 'origin_balance_before', 'origin_balance_after', 'destination_balance_before', 'destination_balance_after']
for col in numerical_cols:
    df_train[[f'{col}_has_decimal', f'{col}_num_decimal_places', f'{col}_num_digits_before_decimal']] = \
        df_train[col].apply(lambda x: pd.Series(extract_decimal_features(x)))
    df_train[f'{col}_has_decimal'] = df_train[f'{col}_has_decimal'].astype(bool)
    df_train[f'{col}_num_decimal_places'] = df_train[f'{col}_num_decimal_places'].astype(int)
    df_train[f'{col}_num_digits_before_decimal'] = df_train[f'{col}_num_digits_before_decimal'].astype(int)

df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df_train['amount'].apply(lambda x: pd.Series(get_first_four_digits(x)))
df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']].fillna(0).astype(int)

# --- Fusion des features de motif de fraude dans df_train ---

# Tri par période : indispensable pour la première apparition et la première fraude
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Nombre de transactions avant la première fraude (comptes destinataires)
num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    account_txns_until_first_fraud = df_sorted_by_period[
        (df_sorted_by_period['destination_account'] == account) &
        (df_sorted_by_period['period'] >= first_appearance) &
        (df_sorted_by_period['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

# Reconstruction de df_fraud_dest_txns pour l'analyse chronologique
# Reprise de fraudulent_destination_accounts
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()
relevant_transactions = df_fraud_dest_txns.copy()

account_timeline_data = []

for account in fraudulent_destination_accounts:
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1

            account_timeline_data.append({
                'account': account,
                'status': current_status,
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

timeline_df = pd.DataFrame(account_timeline_data)

summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)


# time_to_first_fraud
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud'
})
df_train = pd.merge(df_train, temp_destination_fraud_metrics, on='destination_account', how='left')
df_train['time_to_first_fraud'] = df_train['time_to_first_fraud'].fillna(0)

# num_transactions_until_first_fraud
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df_train = pd.merge(df_train, temp_num_txns_until_first_fraud_df, on='destination_account', how='left')
df_train['num_transactions_until_first_fraud'] = df_train['num_transactions_until_first_fraud'].fillna(0)

# Statistiques agrégées par segment
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0)

aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

df_train = pd.merge(df_train, aggregated_segment_stats, on='destination_account', how='left')
df_train[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df_train[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# num_fraud_status_changes
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']
df_train = pd.merge(df_train, num_status_changes_df, on='destination_account', how='left')
df_train['num_fraud_status_changes'] = df_train['num_fraud_status_changes'].fillna(0)

print(f"Shape of df_train after all feature engineering: {df_train.shape}")
print("New features have been added to the training DataFrame.")

# --- Entraînement et évaluation ---
y = df_train['fraud_flag']
X = df_train.drop(columns=['id', 'operation', 'origin_account', 'destination_account',
                           'fraud_flag', 'origin_account_stripped', 'destination_account_stripped',
                           'calc_origin_ipv4', 'calc_origin_mac', 'calc_origin_timestamp',
                           'calc_destination_ipv4', 'calc_destination_mac', 'calc_destination_timestamp',
                           'destination_account_previously_fraud']) # retrait de la feature qui fuit

# Ajout de la feature sans fuite
X['predicted_dest_ever_fraud_proba'] = df_train['predicted_dest_ever_fraud_proba']

# NaN résiduels des conversions hexadécimales
X = X.fillna(0) # NaN remplis à 0

print(f"Using {X.shape[1]} features for training.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost avec scale_pos_weight
    model_fraud_flag = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=42, scale_pos_weight=scale_pos_weight_value)
    model_fraud_flag.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_fraud_flag.predict(X_test)
    y_pred_proba = model_fraud_flag.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for Fraud Flag Prediction (Filtered Data) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'] Bed to Bed , Bed to Bed - Bed to Bed',
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for XGBoost Fraud Prediction (Filtered Data)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    globals()['model_fraud_flag'] = model_fraud_flag
    globals()['X_train'] = X_train
    globals()['y_train'] = y_train
    globals()['X_test'] = X_test
    globals()['y_test'] = y_test
    globals()['y_pred_proba'] = y_pred_proba

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Cible du sous-modèle : le compte destinataire a-t-il déjà fraudé ?
Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df[Y_EVER_FRAUD_TARGET]

# Features pour prédire destination_account_previously_fraud
# On exclut la cible elle-même, fraud_flag, et toute feature encodant
# implicitement une connaissance future de la fraude (features de motif
# calculées sur l'ensemble du jeu de données).
final_features_for_ever_fraud_X = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'calc_origin_decimal',
    'calc_destination_decimal',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_ranked',
    'destination_account_ranked',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1',
    'amount_digit2',
    'amount_digit3',
    'amount_digit4'
]

# On ne garde que les features présentes dans df
final_features_for_ever_fraud_X = [f for f in final_features_for_ever_fraud_X if f in df.columns]

X_ever_fraud = df[final_features_for_ever_fraud_X]

# NaN remplis à 0
X_ever_fraud = X_ever_fraud.fillna(0)

print(f"Features used for 'ever fraudulent' prediction ({len(final_features_for_ever_fraud_X)}): {final_features_for_ever_fraud_X}")
print(f"Shape of X_ever_fraud: {X_ever_fraud.shape}, Shape of y_ever_fraud: {y_ever_fraud.shape}")

if X_ever_fraud.empty:
    print("After data cleaning, no entries remain for training the 'destination account ever fraudulent' model.")
elif len(y_ever_fraud.unique()) < 2:
    print(f"Only one class present in the target variable ('{Y_EVER_FRAUD_TARGET}'). Cannot train a classifier. Unique classes: {y_ever_fraud.unique()}")
else:
    # Séparation train / test du sous-modèle
    X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
        X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
    )

    # scale_pos_weight pour le déséquilibre des classes
    neg_count_ef = y_train_ef.value_counts()[0]
    pos_count_ef = y_train_ef.value_counts()[1]
    scale_pos_weight_ef = neg_count_ef / pos_count_ef
    print(f"Calculated scale_pos_weight for 'ever fraudulent' model: {scale_pos_weight_ef:.2f}")

    # Entraînement XGBoost du sous-modèle
    model_ever_fraud_dest = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr', # Using Average Precision for evaluation
        use_label_encoder=False,
        random_state=42,
        scale_pos_weight=scale_pos_weight_ef
    )
    model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

    # Prédiction sur le test du sous-modèle
    y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
    y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

    # Évaluation du sous-modèle
    accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
    precision_ef = precision_score(y_test_ef, y_pred_ef)
    recall_ef = recall_score(y_test_ef, y_pred_ef)
    roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
    average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

    print(f"\n--- Model Performance for Predicting 'Destination Account Ever Fraudulent' ---")
    print(f"Accuracy: {accuracy_ef:.4f}")
    print(f"Precision: {precision_ef:.4f}")
    print(f"Recall: {recall_ef:.4f}")
    print(f"ROC AUC: {roc_auc_ef:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

    # Matrice de confusion du sous-modèle
    cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
                yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
    plt.title('Confusion Matrix for Ever Fraudulent Destination Account Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Génération de la feature (probabilité) sur tout df
    # Elle servira de feature au modèle principal de détection de fraude
    X_full_for_new_feature_prediction = df[final_features_for_ever_fraud_X].fillna(0)
    df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_full_for_new_feature_prediction)[:, 1]

    print("\nNew feature 'predicted_dest_ever_fraud_proba' added to the original DataFrame 'df'.")
    print(df[['id', 'destination_account', 'destination_account_previously_fraud', 'predicted_dest_ever_fraud_proba', 'fraud_flag']].head())

    globals()['model_ever_fraud_dest'] = model_ever_fraud_dest
    globals()['features_for_ever_fraud_prediction_model'] = final_features_for_ever_fraud_X

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'model_ever_fraud_dest' in globals() and 'features_for_ever_fraud_prediction_model' in globals():
    # Importance des features du sous-modèle XGBoost
    feature_importances_sub_model = model_ever_fraud_dest.feature_importances_
    feature_names_sub_model = features_for_ever_fraud_prediction_model

    # Mise en DataFrame pour l'affichage
    importance_df_sub_model = pd.DataFrame({
        'Feature': feature_names_sub_model,
        'Importance': feature_importances_sub_model
    })

    # Tri par importance décroissante
    importance_df_sub_model = importance_df_sub_model.sort_values(by='Importance', ascending=False)

    # Importance des features
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_sub_model, palette='viridis')
    plt.title('Feature Importance for predicted_dest_ever_fraud_proba Model')
    plt.xlabel('Importance (F-score)')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Erreur : model_ever_fraud_dest ou features_for_ever_fraud_prediction_model introuvable. Exécuter d'abord la cellule d'entraînement du sous-modèle.")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

# Reconstruction de df_train depuis df_filtered_for_training

df_train_main = df_filtered_for_training.copy()

# Cible
y = df_train_main['fraud_flag']

# Features de base du modèle principal :
# les features sans fuite du sous-modèle model_ever_fraud_dest,
# plus predicted_dest_ever_fraud_proba.

final_features_for_ever_fraud_X = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'calc_origin_decimal',
    'calc_destination_decimal',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_ranked',
    'destination_account_ranked',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1',
    'amount_digit2',
    'amount_digit3',
    'amount_digit4'
]

# Ajout de la feature sans fuite
main_model_features = final_features_for_ever_fraud_X + ['predicted_dest_ever_fraud_proba']

# On ne garde que les features présentes dans df_train_main
main_model_features = [f for f in main_model_features if f in df_train_main.columns]

X = df_train_main[main_model_features]

# NaN résiduels des conversions hexadécimales
X = X.fillna(0)

print(f"Using {X.shape[1]} features for training the main model.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the main model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight for main model: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost avec scale_pos_weight
    model_main_fraud = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=42, scale_pos_weight=scale_pos_weight_value)
    model_main_fraud.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_main_fraud.predict(X_test)
    y_pred_proba = model_main_fraud.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- Main XGBoost Model Performance (Non-Leaky Features) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for Main XGBoost Model (Non-Leaky Features)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    globals()['model_main_fraud'] = model_main_fraud
    globals()['X_train_main'] = X_train
    globals()['y_train_main'] = y_train
    globals()['X_test_main'] = X_test
    globals()['y_test_main'] = y_test
    globals()['y_pred_proba_main'] = y_pred_proba

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Importance des features du modèle XGBoost
feature_importances_xgb = model_fraud_flag.feature_importances_
feature_names_xgb = X_train.columns

# Mise en DataFrame pour l'affichage
importance_df_xgb = pd.DataFrame({
    'Feature': feature_names_xgb,
    'Importance': feature_importances_xgb
})

# Tri par importance décroissante
importance_df_xgb = importance_df_xgb.sort_values(by='Importance', ascending=False)

# Importance des features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_xgb, palette='viridis')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance (F-score)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### bonne  piste

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tri par période : indispensable pour la première apparition et la première fraude
df_sorted_by_period = df.sort_values(by='period').copy()

# --- Première apparition et première période frauduleuse ---
def get_fraud_time_metrics(account_type_col, df_data):
    # Comptes de ce type ayant déjà fraudé
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()

    # Accumulateurs
    time_to_fraud_list = []

    for account in fraudulent_accounts:
        # Transactions du compte courant
        account_txns = df_data[df_data[account_type_col] == account]

        # Période de la toute première apparition
        first_appearance_period = account_txns['period'].min()

        # Période de la première transaction frauduleuse
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()

        # Délai jusqu'à la première fraude (0 si la première transaction est déjà une fraude)
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })

    return pd.DataFrame(time_to_fraud_list)

# --- Comptes émetteurs ---
origin_fraud_metrics = get_fraud_time_metrics('origin_account', df_sorted_by_period)

# --- Comptes destinataires ---
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# --- Distributions ---

# Comptes émetteurs
plt.figure(figsize=(10, 6))
sns.histplot(origin_fraud_metrics['time_to_fraud'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Time Elapsed until First Fraud (Origin Accounts)')
plt.xlabel('Periods Elapsed Since First Appearance to First Fraud')
plt.ylabel('Number of Origin Accounts')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Comptes destinataires
plt.figure(figsize=(10, 6))
sns.histplot(destination_fraud_metrics['time_to_fraud'], bins=30, kde=True, color='lightcoral')
plt.title('Distribution of Time Elapsed until First Fraud (Destination Accounts)')
plt.xlabel('Periods Elapsed Since First Appearance to First Fraud')
plt.ylabel('Number of Destination Accounts')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("Descriptive statistics for time to fraud (Origin Accounts):")
print(origin_fraud_metrics['time_to_fraud'].describe())
print("\nDescriptive statistics for time to fraud (Destination Accounts):")
print(destination_fraud_metrics['time_to_fraud'].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Reprise de fraudulent_destination_accounts
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Transactions dont le compte destinataire est frauduleux
df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Comptage par destination_account et fraud_flag
fraud_distribution_per_dest = df_fraud_dest_txns.groupby(['destination_account', 'fraud_flag']).size().unstack(fill_value=0)

# Renommage des colonnes
fraud_distribution_per_dest.columns = ['non_fraudulent_transactions', 'fraudulent_transactions']

# Nombre total de transactions par compte
fraud_distribution_per_dest['total_transactions'] = fraud_distribution_per_dest['non_fraudulent_transactions'] + fraud_distribution_per_dest['fraudulent_transactions']

# Proportion de transactions frauduleuses par compte
fraud_distribution_per_dest['fraud_ratio'] = fraud_distribution_per_dest['fraudulent_transactions'] / fraud_distribution_per_dest['total_transactions']

print("\nDistribution des transactions (frauduleuses vs. non-frauduleuses) pour chaque compte destinataire frauduleux:")
display(fraud_distribution_per_dest.head())

print("\nStatistiques descriptives du ratio de fraude par compte destinataire frauduleux:")
display(fraud_distribution_per_dest['fraud_ratio'].describe())

# Distribution des transactions frauduleuses et non frauduleuses
plt.figure(figsize=(14, 7))
sns.histplot(fraud_distribution_per_dest['fraud_ratio'], bins=30, kde=True, color='skyblue')
plt.title('Distribution du Ratio de Transactions Frauduleuses pour les Comptes Destinataires Frauduleux')
plt.xlabel('Ratio de Transactions Frauduleuses (Fraudulent / Total)')
plt.ylabel('Nombre de Comptes Destinataires')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Comptages bruts pour quelques comptes
# Top 5 des comptes par ratio de fraude
top_fraud_accounts = fraud_distribution_per_dest.sort_values(by='fraud_ratio', ascending=False).head(5)

if not top_fraud_accounts.empty:
    print("\nVisualisation des comptes destinataires avec le ratio de fraude le plus élevé:")
    top_fraud_accounts[['non_fraudulent_transactions', 'fraudulent_transactions']].plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
    plt.title('Nombre de Transactions Frauduleuses et Non-Frauduleuses pour les Top Comptes Destinataires Frauduleux')
    plt.xlabel('Compte Destinataire')
    plt.ylabel('Nombre de Transactions')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Type de Transaction')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Transactions dont le compte destinataire a déjà fraudé

relevant_transactions = df_fraud_dest_txns.copy()

# Analyse détaillée par compte
account_timeline_data = []

# Parcours des comptes destinataires frauduleux
for account in fraudulent_destination_accounts:
    # Transactions du compte courant, triées par période
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    # Initialisation du premier segment
    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    # Détection des changements de statut
    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            # Fin du segment précédent
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1 # période incluse

            account_timeline_data.append({
                'account': account,
                'status': current_status, # 0 = non-fraude, 1 = fraude
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            # Nouveau segment
            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    # Dernier segment après la boucle
    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

# Passage en DataFrame
timeline_df = pd.DataFrame(account_timeline_data)

print("Analyse de la chronologie des statuts de fraude par compte destinataire:")
display(timeline_df.head())

# Synthèse des segments frauduleux (statut=1) et non frauduleux (statut=0)
summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

print("\nStatistiques récapitulatives des phases (frauduleuses vs. non-frauduleuses):")
display(summary_stats)

# Distributions de la durée et du nombre de transactions par segment
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
sns.histplot(timeline_df[timeline_df['status'] == 0]['duration'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['duration'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution des Durées de Phase par Statut de Fraude')
plt.xlabel('Durée (Périodes)')
plt.ylabel('Fréquence')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(timeline_df[timeline_df['status'] == 0]['num_transactions'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['num_transactions'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution du Nombre de Transactions par Statut de Fraude')
plt.xlabel('Nombre de Transactions')
plt.ylabel('Fréquence')
plt.legend()

plt.tight_layout()
plt.show()

# Nombre de changements de statut par compte
# Un changement = fraud_flag qui change d'une transaction à la suivante
# Sur le df complet trié par période, groupé par compte

# Colonne intermédiaire : variation de fraud_flag dans la chronologie du compte
relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

# Comptage des changements réels (0 vers 1 ou 1 vers 0)
# Un diff non nul indique un changement de statut
num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

print("\nDistribution du nombre de changements de statut (Non-Fraude <-> Fraude) par compte:")
# Comptes sans changement (un seul segment) : 0 changement
all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)

display(num_status_changes_per_account_full.describe())

# Distribution du nombre de changements de statut
plt.figure(figsize=(10, 6))
# Bins de 0 à max_changes + 1 pour un comptage par valeur entière
bins = np.arange(num_status_changes_per_account_full.max() + 2) - 0.5
sns.histplot(num_status_changes_per_account_full, bins=bins, kde=False)
plt.title('Nombre de Changements de Statut par Compte Destinataire Frauduleux')
plt.xlabel('Nombre de Changements de Statut')
plt.ylabel('Nombre de Comptes')
plt.xticks(np.arange(0, num_status_changes_per_account_full.max() + 1, 1))
plt.show()

In [ ]:
import pandas as pd
import numpy as np


# --- Nombre de transactions avant la première fraude (comptes destinataires) ---

# destination_fraud_metrics contient first_appearance_period et first_fraud_period
# pour chaque compte destinataire ayant déjà fraudé.

num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    # Transactions du compte jusqu'à la période de première fraude incluse
    account_txns_until_first_fraud = df[
        (df['destination_account'] == account) &
        (df['period'] >= first_appearance) &
        (df['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

print("\n--- Delai avant la premiere fraude ---")

print("\nPartie 1 : combien de periodes et de transactions avant qu'un compte destinataire soit declare frauduleux ?")
print("-"*110)

print("Statistiques du delai jusqu'a la premiere fraude (en periodes), comptes destinataires :")
display(destination_fraud_metrics['time_to_fraud'].describe())

print("Statistiques du nombre de transactions jusqu'a la premiere fraude, comptes destinataires :")
display(num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].describe())

print("\nLecture :")
print(f"- Delai moyen jusqu'a la premiere fraude : {destination_fraud_metrics['time_to_fraud'].mean():.2f} periodes, 75e centile a {destination_fraud_metrics['time_to_fraud'].quantile(0.75):.0f} periodes.")
print(f"- Nombre moyen de transactions jusqu'a la premiere fraude : {num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].mean():.2f}, 75e centile a {num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].quantile(0.75):.0f} transactions.")

print("\nPartie 2 : duree et nombre de transactions des phases frauduleuses et non frauduleuses.")
print("-"*110)

print("Statistiques des segments frauduleux et non frauduleux :")
display(summary_stats)

print("\nLecture :")
print(f"- Duree moyenne des segments non frauduleux (statut=0) : {summary_stats.loc[0, 'mean_duration']:.2f} periodes et {summary_stats.loc[0, 'mean_transactions']:.2f} transactions. Segments frauduleux (statut=1) : {summary_stats.loc[1, 'mean_duration']:.2f} periodes et {summary_stats.loc[1, 'mean_transactions']:.2f} transactions. La duree et le nombre de transactions baissent nettement une fois le compte en phase frauduleuse.")
print("- Les histogrammes montrent toutefois un fort recouvrement des distributions de duree et de nombre de transactions entre segments frauduleux et non frauduleux : pour beaucoup de comptes, les deux profils restent proches.")


In [ ]:
import numpy as np

# 1. Fusion de time_to_fraud pour les comptes destinataires
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud' # Rename the column here
})
df = pd.merge(
    df,
    temp_destination_fraud_metrics,
    on='destination_account',
    how='left'
)
df['time_to_first_fraud'] = df['time_to_first_fraud'].fillna(0) # NaN pour les comptes sans fraude

# 2. Fusion de num_transactions_until_first_fraud pour les comptes destinataires
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df = pd.merge(
    df,
    temp_num_txns_until_first_fraud_df,
    on='destination_account',
    how='left'
)
df['num_transactions_until_first_fraud'] = df['num_transactions_until_first_fraud'].fillna(0)

# 3. Statistiques agrégées par segment depuis timeline_df
# Durée et nombre de transactions moyens des segments frauduleux (1) et non frauduleux (0)
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0) # fill_value=0 si le statut n'existe pas pour un compte

# Aplatissement des colonnes multi-niveaux
aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

# Renommage des colonnes
aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

# Fusion des statistiques de segment dans le df principal
df = pd.merge(
    df,
    aggregated_segment_stats,
    on='destination_account',
    how='left'
)
# NaN pour les comptes absents de timeline_df
df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# 4. Fusion du nombre de changements de statut par compte
# num_status_changes_per_account_full est une Series : passage en DataFrame pour la fusion
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']

df = pd.merge(
    df,
    num_status_changes_df,
    on='destination_account',
    how='left'
)
df['num_fraud_status_changes'] = df['num_fraud_status_changes'].fillna(0)

print("New features based on fraud patterns have been added to the DataFrame.")
print(df[['id', 'destination_account', 'time_to_first_fraud', 'num_transactions_until_first_fraud',
          'mean_fraud_duration', 'mean_fraud_transactions', 'num_fraud_status_changes', 'fraud_flag']].head())

In [ ]:
import numpy as np
import pandas as pd

# --- Fusion des features nécessaires dans df ---

# Tri par période : indispensable pour la première apparition et la première fraude
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Nombre de transactions avant la première fraude (comptes destinataires)
num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    account_txns_until_first_fraud = df_sorted_by_period[
        (df_sorted_by_period['destination_account'] == account) &
        (df_sorted_by_period['period'] >= first_appearance) &
        (df_sorted_by_period['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

# Reconstruction de df_fraud_dest_txns pour l'analyse chronologique
# Reprise de fraudulent_destination_accounts
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()
relevant_transactions = df_fraud_dest_txns.copy()

account_timeline_data = []

for account in fraudulent_destination_accounts:
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1

            account_timeline_data.append({
                'account': account,
                'status': current_status,
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

timeline_df = pd.DataFrame(account_timeline_data)

relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)


# 1. Fusion de time_to_fraud pour les comptes destinataires
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud' # Rename the column here
})
df = pd.merge(
    df,
    temp_destination_fraud_metrics,
    on='destination_account',
    how='left'
)
df['time_to_first_fraud'] = df['time_to_first_fraud'].fillna(0) # NaN pour les comptes sans fraude

# 2. Fusion de num_transactions_until_first_fraud pour les comptes destinataires
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df = pd.merge(
    df,
    temp_num_txns_until_first_fraud_df,
    on='destination_account',
    how='left'
)
df['num_transactions_until_first_fraud'] = df['num_transactions_until_first_fraud'].fillna(0)

# 3. Statistiques agrégées par segment depuis timeline_df
# Durée et nombre de transactions moyens des segments frauduleux (1) et non frauduleux (0)
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0) # fill_value=0 si le statut n'existe pas pour un compte

# Aplatissement des colonnes multi-niveaux
aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

# Renommage des colonnes
aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

# Fusion des statistiques de segment dans le df principal
df = pd.merge(
    df,
    aggregated_segment_stats,
    on='destination_account',
    how='left'
)
# NaN pour les comptes absents de timeline_df
df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# 4. Fusion du nombre de changements de statut par compte
# num_status_changes_per_account_full est une Series : passage en DataFrame pour la fusion
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']

df = pd.merge(
    df,
    num_status_changes_df,
    on='destination_account',
    how='left'
)
df['num_fraud_status_changes'] = df['num_fraud_status_changes'].fillna(0)



# 1. p_geometric_first_fraud_txns
# Basée sur le nombre de transactions avant la première fraude du compte
df['p_geometric_first_fraud_txns'] = 1 / (df['num_transactions_until_first_fraud'] + 1)

# 2. p_geometric_non_fraud_exit_period
# Probabilité géométrique de sortir d'une phase non frauduleuse à la période suivante
df['p_geometric_non_fraud_exit_period'] = 1 / (df['mean_non_fraud_duration'] + 1)

# 3. p_geometric_non_fraud_exit_txns
# Probabilité géométrique de sortir d'une phase non frauduleuse à la transaction suivante
df['p_geometric_non_fraud_exit_txns'] = 1 / (df['mean_non_fraud_transactions'] + 1)

print("New geometric features have been added to the DataFrame:")
print(df[[
    'destination_account',
    'num_transactions_until_first_fraud',
    'p_geometric_first_fraud_txns',
    'mean_non_fraud_duration',
    'p_geometric_non_fraud_exit_period',
    'mean_non_fraud_transactions',
    'p_geometric_non_fraud_exit_txns',
    'fraud_flag'
]].head())


### Temps avant la première fraude : émetteurs vs destinataires

Courbes de densité (KDE) superposées pour les deux rôles, afin de comparer le délai entre la première apparition d'un compte et sa première transaction frauduleuse.

In [ ]:
plt.figure(figsize=(12, 7))
sns.kdeplot(origin_fraud_metrics['time_to_fraud'], fill=True, color='skyblue', label='Comptes Émetteurs')
sns.kdeplot(destination_fraud_metrics['time_to_fraud'], fill=True, color='lightcoral', label='Comptes Destinataires')

plt.title('Distribution Comparée du Temps Écoulé jusqu\'à la Première Fraude')
plt.xlabel('Périodes Écoulées Depuis la Première Apparition jusqu\'à la Première Fraude')
plt.ylabel('Densité')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

la premiere fois qu'un compte est declaré frauduleux c'est environ vers sa 3 eme apparution.

Chiffres : le délai moyen entre la première apparition d'un compte destinataire et sa première fraude est de **2,19 périodes**, et 75 % des comptes destinataires frauduleux fraudent dans les **3 périodes** qui suivent leur première apparition. Un compte destinataire bascule donc vite après son entrée en scène.

### Utilisation de la Loi Géométrique pour la Détection de Fraude

La loi géométrique modélise le nombre d'essais de Bernoulli nécessaires pour obtenir le premier succès. Dans notre contexte, si la 'durée jusqu'à la première fraude' (`time_to_fraud`) pour un compte suit une loi géométrique, nous pouvons estimer la probabilité d'un événement de fraude ('p') à chaque période, en supposant que le compte n'a pas encore été frauduleux.

Le temps moyen d'une distribution géométrique (nombre d'essais, incluant le succès) est de `1/p`. Étant donné que `time_to_fraud` représente le nombre de périodes *avant* la première fraude (où `time_to_fraud = 0` signifie fraude dès la première apparition), le nombre total de périodes jusqu'à la première fraude est `time_to_fraud + 1`.

Nous pouvons donc estimer le paramètre `p` comme suit :
`p = 1 / (moyenne_time_to_fraud + 1)`

Ce paramètre `p` représente la probabilité qu'un compte destinataire devienne frauduleux dans la prochaine période, étant donné qu'il n'a pas été frauduleux jusqu'à présent depuis sa première apparition. Cette valeur peut être une caractéristique discriminante très utile pour le modèle.

### `p_geometric_first_fraud` sépare-t-elle les classes ?

Distribution de la feature pour les transactions frauduleuses et non frauduleuses. Une séparation nette signalerait une feature discriminante.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Données du tracé de p_geometric_first_fraud
df_p_geo_plot = pd.DataFrame({
    'p_geometric_first_fraud': X_test_enh['p_geometric_first_fraud'],
    'fraud_flag': y_test_enh
})

plt.figure(figsize=(10, 6))
sns.boxplot(x='fraud_flag', y='p_geometric_first_fraud', data=df_p_geo_plot, palette='coolwarm')
plt.title('Distribution de p_geometric_first_fraud par Fraud Flag (Test Set)')
plt.xlabel('Fraud Flag (0 = Non-Fraude, 1 = Fraude)')
plt.ylabel('Probabilité Géométrique de Première Fraude (p_geometric_first_fraud)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_p_geo_plot, x='p_geometric_first_fraud', hue='fraud_flag', fill=True, common_norm=False, palette='coolwarm')
plt.title('Densité de p_geometric_first_fraud par Fraud Flag (Test Set)')
plt.xlabel('Probabilité Géométrique de Première Fraude (p_geometric_first_fraud)')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### Scores de fraude : modèle de référence vs modèle enrichi

Distributions des probabilités prédites, avec et sans `p_geometric_first_fraud`. Si la feature apporte, le modèle enrichi remonte les vraies fraudes et fait redescendre les non-fraudes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Données du tracé des probabilités prédites
df_proba_plot = pd.DataFrame({
    'Actual Fraud Flag': y_test_enh, # y_test_base et y_test_enh sont identiques
    'Baseline Predicted Proba': y_pred_proba_base,
    'Enhanced Predicted Proba': y_pred_proba_enh
})

# Distribution des probabilités prédites, modèle de référence
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_proba_plot, x='Baseline Predicted Proba', hue='Actual Fraud Flag', fill=True, common_norm=False, palette='viridis')
plt.title('Distribution des Probabilités de Fraude Prédites (Modèle Baseline)')
plt.xlabel('Probabilité Prédite de Fraude')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Distribution des probabilités prédites, modèle enrichi
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_proba_plot, x='Enhanced Predicted Proba', hue='Actual Fraud Flag', fill=True, common_norm=False, palette='viridis')
plt.title('Distribution des Probabilités de Fraude Prédites (Modèle Amélioré avec p_geometric_first_fraud)')
plt.xlabel('Probabilité Prédite de Fraude')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Tri par période : indispensable pour la première apparition et la première fraude
df_sorted_by_period = df.sort_values(by='period').copy()

# Première apparition et première période frauduleuse
def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

# Génération de destination_fraud_metrics
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Paramètre p de la loi géométrique, à partir de mean_time_to_fraud
mean_time_to_fraud_dest = destination_fraud_metrics['time_to_fraud'].mean()
p_geometric = 1 / (mean_time_to_fraud_dest + 1)

print(f"Le paramètre 'p' de la loi géométrique pour le 'time_to_fraud' des comptes destinataires est : {p_geometric:.4f}")

# Cette valeur de p devient une feature du compte destinataire
# Feature statique par compte, fondée sur son comportement avant la fraude

destination_fraud_metrics['p_geometric_first_fraud'] = destination_fraud_metrics['time_to_fraud'].apply(lambda x: 1 / (x + 1) if x >= 0 else 0) # garde-fou contre un time_to_fraud négatif

# Fusion de p_geometric_first_fraud dans df
df = pd.merge(df, destination_fraud_metrics[['account', 'p_geometric_first_fraud']].rename(columns={'account': 'destination_account'}), on='destination_account', how='left')
df['p_geometric_first_fraud'] = df['p_geometric_first_fraud'].fillna(0) # valeur par défaut pour les comptes non frauduleux

print("\nFirst 5 rows of destination_fraud_metrics with the new 'p_geometric_first_fraud' feature:")
display(destination_fraud_metrics.head())
print("\nFirst 5 rows of df with the new 'p_geometric_first_fraud' feature:")
display(df[['id', 'destination_account', 'p_geometric_first_fraud']].head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Model Performance Comparison: Baseline vs. Enhanced with p_geometric_first_fraud ---")

# Cible
y = df['fraud_flag']

# Features de référence
baseline_features = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'origin_account_previously_fraud',
    'destination_account_previously_fraud'
]

# Vérification de leur présence dans le DataFrame
baseline_features = [f for f in baseline_features if f in df.columns]

# --- Modèle de référence ---
print("\n--- Training Baseline Model ---")
X_baseline = df[baseline_features]
X_baseline = X_baseline.fillna(0) # NaN remplis à 0

# Séparation train / test, modèle de référence
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline, y, test_size=0.3, random_state=42, stratify=y
)

# RandomForest de référence
rf_baseline = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
rf_baseline.fit(X_train_base, y_train_base)

# Prédiction
y_pred_base = rf_baseline.predict(X_test_base)
y_pred_proba_base = rf_baseline.predict_proba(X_test_base)[:, 1]

# Évaluation du modèle de référence
accuracy_base = accuracy_score(y_test_base, y_pred_base)
precision_base = precision_score(y_test_base, y_pred_base)
recall_base = recall_score(y_test_base, y_pred_base)
f1_base = f1_score(y_test_base, y_pred_base)
roc_auc_base = roc_auc_score(y_test_base, y_pred_proba_base)

print(f"Baseline Model Performance:")
print(f"  Accuracy: {accuracy_base:.4f}")
print(f"  Precision: {precision_base:.4f}")
print(f"  Recall: {recall_base:.4f}")
print(f"  F1-Score: {f1_base:.4f}")
print(f"  ROC AUC: {roc_auc_base:.4f}")

# Matrice de confusion, modèle de référence
cm_base = confusion_matrix(y_test_base, y_pred_base)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_base, display_labels=['Non-Fraud', 'Fraud']).plot(cmap='Blues')
plt.title('Baseline Model Confusion Matrix')
plt.show()

# --- Modèle enrichi de p_geometric_first_fraud ---
print("\n--- Training Enhanced Model (with p_geometric_first_fraud) ---")
enhanced_features = baseline_features + ['p_geometric_first_fraud']

# Vérification de la présence des features enrichies
enhanced_features = [f for f in enhanced_features if f in df.columns]

X_enhanced = df[enhanced_features]
X_enhanced = X_enhanced.fillna(0) # NaN remplis à 0

# Séparation train / test, modèle enrichi
X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    X_enhanced, y, test_size=0.3, random_state=42, stratify=y
)

# RandomForest enrichi
rf_enhanced = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
rf_enhanced.fit(X_train_enh, y_train_enh)

# Prédiction
y_pred_enh = rf_enhanced.predict(X_test_enh)
y_pred_proba_enh = rf_enhanced.predict_proba(X_test_enh)[:, 1]

# Évaluation du modèle enrichi
accuracy_enh = accuracy_score(y_test_enh, y_pred_enh)
precision_enh = precision_score(y_test_enh, y_pred_enh)
recall_enh = recall_score(y_test_enh, y_pred_enh)
f1_enh = f1_score(y_test_enh, y_pred_enh)
roc_auc_enh = roc_auc_score(y_test_enh, y_pred_proba_enh)

print(f"Enhanced Model Performance:")
print(f"  Accuracy: {accuracy_enh:.4f}")
print(f"  Precision: {precision_enh:.4f}")
print(f"  Recall: {recall_enh:.4f}")
print(f"  F1-Score: {f1_enh:.4f}")
print(f"  ROC AUC: {roc_auc_enh:.4f}")

# Matrice de confusion, modèle enrichi
cm_enh = confusion_matrix(y_test_enh, y_pred_enh)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_enh, display_labels=['Non-Fraud', 'Fraud']).plot(cmap='Blues')
plt.title('Enhanced Model Confusion Matrix')
plt.show()

# --- Comparaison ---
print("\n--- Performance Comparison ---")
print(f"Metric        | Baseline | Enhanced | Change")
print(f"----------------------------------------------")
print(f"Accuracy      | {accuracy_base:.4f}   | {accuracy_enh:.4f}   | {accuracy_enh - accuracy_base:+.4f}")
print(f"Precision     | {precision_base:.4f}   | {precision_enh:.4f}   | {precision_enh - precision_base:+.4f}")
print(f"Recall        | {recall_base:.4f}   | {recall_enh:.4f}   | {recall_enh - recall_base:+.4f}")
print(f"F1-Score      | {f1_base:.4f}   | {f1_enh:.4f}   | {f1_enh - f1_base:+.4f}")
print(f"ROC AUC       | {roc_auc_base:.4f}   | {roc_auc_enh:.4f}   | {roc_auc_enh - roc_auc_base:+.4f}")

print("\nObservation: The inclusion of 'p_geometric_first_fraud' has resulted in a change in model performance. Review the metrics above to understand the specific impact.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Importance des features du modèle enrichi
feature_importances_enh = rf_enhanced.feature_importances_
feature_names_enh = X_enhanced.columns

# Mise en DataFrame pour l'affichage
importance_df_enh = pd.DataFrame({
    'Feature': feature_names_enh,
    'Importance': feature_importances_enh
})

# Tri par importance décroissante
importance_df_enh = importance_df_enh.sort_values(by='Importance', ascending=False)

# Importance des features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=importance_df_enh, palette='viridis', legend=False)
plt.title('Enhanced Model Feature Importance')
plt.xlabel('Importance (Mean Decrease in Impurity)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## old

In [ ]:
import numpy as np

numerical_cols = [
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after'
]

# Propriétés décimales du montant
def extract_decimal_features(value):
    if pd.isna(value):
        return False, 0, 0
    s = str(value)

    # Gestion de la notation scientifique
    if 'e' in s or 'E' in s:
        # Passage en notation décimale standard
        s = f"{value:.10f}" # précision raisonnable
        s = s.rstrip('0').rstrip('.') # Remove trailing zeros and decimal if not needed

    has_decimal = '.' in s

    if has_decimal:
        parts = s.split('.')
        integer_part = parts[0]
        decimal_part = parts[1]
        num_decimal_places = len(decimal_part)
    else:
        integer_part = s
        num_decimal_places = 0

    num_digits_before_decimal = len(integer_part.lstrip('-'))

    return has_decimal, num_decimal_places, num_digits_before_decimal


for col in numerical_cols:
    # Création des colonnes
    df[[f'{col}_has_decimal', f'{col}_num_decimal_places', f'{col}_num_digits_before_decimal']] = \
        df[col].apply(lambda x: pd.Series(extract_decimal_features(x)))

    # Typage booléen et entier
    df[f'{col}_has_decimal'] = df[f'{col}_has_decimal'].astype(bool)
    df[f'{col}_num_decimal_places'] = df[f'{col}_num_decimal_places'].astype(int)
    df[f'{col}_num_digits_before_decimal'] = df[f'{col}_num_digits_before_decimal'].astype(int)

# Quatre colonnes pour les quatre premiers chiffres de amount
def get_first_four_digits(value):
    if pd.isna(value):
        return [np.nan] * 4
    s = str(value)

    # Gestion de la notation scientifique
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}" # notation décimale standard

    # Suppression du signe et de la virgule
    s_digits = s.replace('.', '').lstrip('-')

    digits = []
    for i in range(4):
        if i < len(s_digits):
            digits.append(int(s_digits[i]))
        else:
            digits.append(np.nan) # NaN pour les chiffres absents
    return digits

df[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df['amount'].apply(lambda x: pd.Series(get_first_four_digits(x)))

print("New decimal and digit-based features created.")
print(df[['amount', 'amount_has_decimal', 'amount_num_decimal_places', 'amount_num_digits_before_decimal', 'amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']].head())


### Nouvelles features et fraude, sur op_03

Distribution de `_has_decimal`, `_num_decimal_places`, `_num_digits_before_decimal` et `amount_digitX` sur les seules transactions `op_03`, croisée avec `fraud_flag`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Restriction à operation == op_03
df_op03 = df[df['operation'] == 'op_03'].copy()

print(f"Filtered DataFrame shape (only op_03): {df_op03.shape}")

#### Caractéristiques liées aux décimales (`_has_decimal`)

In [ ]:
decimal_bool_cols = [
    'amount_has_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_after_has_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_after_has_decimal'
]

for col in decimal_bool_cols:
    plt.figure(figsize=(8, 5))
    sns.countplot(x=col, hue='fraud_flag', data=df_op03, palette='pastel')
    plt.title(f'Distribution du Fraud Flag par {col} pour op_03')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Fraud Flag')
    plt.show()

#### Caractéristiques liées au nombre de décimales (`_num_decimal_places`)

In [ ]:
num_decimal_places_cols = [
    'amount_num_decimal_places',
    'origin_balance_before_num_decimal_places',
    'origin_balance_after_num_decimal_places',
    'destination_balance_before_num_decimal_places',
    'destination_balance_after_num_decimal_places'
]

for col in num_decimal_places_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='fraud_flag', y=col, data=df_op03, palette='viridis')
    plt.title(f'Distribution de {col} par Fraud Flag pour op_03')
    plt.xlabel('Fraud Flag')
    plt.ylabel(col)
    plt.show()

#### Caractéristiques liées au nombre de chiffres avant la virgule (`_num_digits_before_decimal`)

In [ ]:
num_digits_before_decimal_cols = [
    'amount_num_digits_before_decimal',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_num_digits_before_decimal'
]

for col in num_digits_before_decimal_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='fraud_flag', y=col, data=df_op03, palette='magma')
    plt.title(f'Distribution de {col} par Fraud Flag pour op_03')
    plt.xlabel('Fraud Flag')
    plt.ylabel(col)
    plt.show()

#### Caractéristiques liées aux quatre premiers chiffres du montant (`amount_digitX`)

In [ ]:
amount_digit_cols = ['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']

for col in amount_digit_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(x=col, hue='fraud_flag', data=df_op03, palette='coolwarm')
    plt.title(f'Distribution du Fraud Flag par {col} pour op_03')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Fraud Flag')
    plt.show()

### Observations

- **`_has_decimal`** : la présence ou l'absence de décimales est liée à `fraud_flag` sur la plupart des colonnes ; à `False`, le nombre de fraudes chute nettement.
- **`_num_decimal_places` et `_num_digits_before_decimal`** : les distributions diffèrent entre fraudes et non-fraudes selon la colonne. Des montants ronds, ou à nombre de chiffres fixe, ressortent côté fraude.
- **`amount_digitX`** : certains premiers chiffres du montant (`amount_digit1` à `amount_digit4`) sont surreprésentés d'un côté ou de l'autre. À rapprocher de la loi de Benford : un écart à la distribution attendue des premiers chiffres trahit souvent une saisie manipulée.

Ces features capturent des motifs invisibles dans les montants et les soldes bruts.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

# Cible
y = df['fraud_flag']

# Features du modèle
# Nouvelles features de décimales et de chiffres, plus les features numériques existantes
features = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'destination_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

# On écarte les features non encore créées
existing_features = [f for f in features if f in df.columns]
X = df[existing_features]

# NaN résiduels des conversions hexadécimales
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in X.columns:
        X[col] = X[col].fillna(0) # NaN remplis à 0

# Suppression des lignes avec NaN résiduels dans les features
X = X.dropna()
y = y[X.index]

print(f"Using {len(existing_features)} features for training.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # scale_pos_weight pour le déséquilibre des classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Entraînement XGBoost avec scale_pos_weight
    model_fraud_flag_stage2 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', use_label_encoder=False, random_state=42, scale_pos_weight=scale_pos_weight_value)
    model_fraud_flag_stage2.fit(X_train, y_train)

    # Prédiction sur le test
    y_pred = model_fraud_flag_stage2.predict(X_test)
    y_pred_proba = model_fraud_flag_stage2.predict_proba(X_test)[:, 1]

    # Évaluation
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for Fraud Flag Prediction (all operations) with Class Weights ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for XGBoost Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

# Courbe précision-rappel
plt.figure(figsize=(8, 6))
# from_estimator si le modèle est un estimateur, sinon from_predictions
display = PrecisionRecallDisplay.from_predictions(y_test, y_pred_proba, name="XGBoost Classifier")
display.plot(ax=plt.gca())
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

### Réglage des hyperparamètres par GridSearchCV

Recherche exhaustive sur une grille de paramètres XGBoost, scorée en Average Precision.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Grille de paramètres
# Hyperparamètres les plus influents pour XGBoost
param_grid = {
    'n_estimators': [100, 200, 300], # nombre d'itérations de boosting
    'max_depth': [3, 5, 7],         # profondeur maximale d'un arbre
    'learning_rate': [0.01, 0.1, 0.2], # pas d'apprentissage, contre le surapprentissage
    'subsample': [0.7, 0.8, 1.0],   # taux d'échantillonnage des lignes
    'colsample_bytree': [0.7, 0.8, 1.0] # taux d'échantillonnage des colonnes par arbre
}

# GridSearchCV
# Métrique aucpr, adaptée aux jeux déséquilibrés
grid_search = GridSearchCV(estimator=model_fraud_flag_stage2, # modèle XGBoost défini plus haut
                           param_grid=param_grid,
                           scoring='average_precision', # Optimize for Average Precision (PR-AUC)
                           cv=3, # validation croisée à 3 plis
                           verbose=2, # affiche la progression
                           n_jobs=-1) # tous les cœurs disponibles

print("Starting GridSearchCV...")
# Recherche sur les données d'entraînement
grid_search.fit(X_train, y_train)
print("GridSearchCV completed.")

In [ ]:
# Meilleurs paramètres trouvés par GridSearchCV
print(f"Best parameters found: {grid_search.best_params_}")

# Meilleur score
print(f"Best Average Precision score: {grid_search.best_score_:.4f}")

### Évaluation du meilleur modèle

Performance sur le test du modèle retenu par la recherche sur grille.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix

# Meilleur estimateur de la recherche
best_model = grid_search.best_estimator_

# Prédiction sur le test avec le meilleur modèle
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Évaluation du meilleur modèle
accuracy_best = accuracy_score(y_test, y_pred_best)
precision_best = precision_score(y_test, y_pred_best)
recall_best = recall_score(y_test, y_pred_best)
roc_auc_best = roc_auc_score(y_test, y_pred_proba_best)
average_precision_best = average_precision_score(y_test, y_pred_proba_best)

print(f"\n--- Tuned XGBoost Model Performance --- ")
print(f"Accuracy: {accuracy_best:.4f}")
print(f"Precision: {precision_best:.4f}")
print(f"Recall: {recall_best:.4f}")
print(f"ROC AUC: {roc_auc_best:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_best:.4f}")

# Matrice de confusion du meilleur modèle
cm_best = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Tuned XGBoost Fraud Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Importance des features — XGBoost

Contribution de chaque feature aux prédictions : plus le score est élevé, plus la feature pèse. Utile pour repérer ce qui porte réellement la détection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'best_model' not in locals() and 'grid_search' in locals():
    best_model = grid_search.best_estimator_
elif 'best_model' not in locals() and 'grid_search' not in locals():
    print("Error: 'best_model' and 'grid_search' are not defined. Please run the XGBoost model training and GridSearchCV cells first.")

if 'X_train' not in locals():
    print("Erreur : X_train non défini. Exécuter d'abord la cellule de séparation train / test.")

# Importance des features du meilleur modèle XGBoost
if 'best_model' in locals() and 'X_train' in locals():
    feature_importances_xgb = best_model.feature_importances_
    feature_names_xgb = X_train.columns

    # Mise en DataFrame pour l'affichage
    importance_df_xgb = pd.DataFrame({
        'Feature': feature_names_xgb,
        'Importance': feature_importances_xgb
    })

    # Tri par importance
    importance_df_xgb = importance_df_xgb.sort_values(by='Importance', ascending=False)

    # Importance des features
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_xgb, palette='viridis')
    plt.title('XGBoost Feature Importance')
    plt.xlabel('Importance (F-score)')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot XGBoost feature importance: Missing required variables.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier

# Restriction aux transactions op_03
df_op03_catboost = df[df['operation'] == 'op_03'].copy()

# On ne garde que les features présentes dans le sous-ensemble op_03
features_for_op03_catboost = [f for f in existing_features if f in df_op03_catboost.columns]

X_op03_catboost = df_op03_catboost[features_for_op03_catboost]
y_op03_catboost = df_op03_catboost['fraud_flag']

# NaN des features sur le sous-ensemble op_03
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in X_op03_catboost.columns:
        X_op03_catboost[col] = X_op03_catboost[col].fillna(0) # NaN remplis à 0

# Suppression des lignes avec NaN résiduels
X_op03_catboost = X_op03_catboost.dropna()
y_op03_catboost = y_op03_catboost[X_op03_catboost.index]

print(f"Using {len(features_for_op03_catboost)} features for CatBoost training on op_03 transactions.")
print(f"Shape of X_op03_catboost: {X_op03_catboost.shape}, Shape of y_op03_catboost: {y_op03_catboost.shape}")

if X_op03_catboost.empty:
    print("After data cleaning, no entries remain for training the CatBoost model on op_03.")
elif len(y_op03_catboost.unique()) < 2:
    print(f"Only one class present in the target variable for op_03 transactions. Cannot train a classifier. Unique classes: {y_op03_catboost.unique()}")
else:
    # Séparation train / test sur op_03
    X_train_op03, X_test_op03, y_train_op03, y_test_op03 = train_test_split(
        X_op03_catboost, y_op03_catboost, test_size=0.3, random_state=42, stratify=y_op03_catboost
    )

    # scale_pos_weight pour le déséquilibre des classes
    neg_count_op03 = y_train_op03.value_counts()[0]
    pos_count_op03 = y_train_op03.value_counts()[1]
    scale_pos_weight_value_op03 = neg_count_op03 / pos_count_op03
    print(f"Calculated scale_pos_weight for op_03: {scale_pos_weight_value_op03:.2f}")

    # Grille de paramètres CatBoost
    catboost_param_grid = {
        'iterations': [100, 200], # équivalent de n_estimators
        'depth': [5, 7],         # équivalent de max_depth
        'learning_rate': [0.05, 0.1],
        'l2_leaf_reg': [1, 3],
        'scale_pos_weight': [scale_pos_weight_value_op03] # scale_pos_weight ajouté à la grille
    }

    # CatBoostClassifier sans scale_pos_weight dans le constructeur
    cat_model = CatBoostClassifier(
        random_seed=42,
        verbose=0, # sortie silencieuse à l'entraînement
        eval_metric='AUC:hints=skip_train~false' # Use AUC for evaluation during CV
    )

    print("\nStarting GridSearchCV for CatBoost on op_03 transactions...")
    grid_search_catboost = GridSearchCV(
        estimator=cat_model,
        param_grid=catboost_param_grid,
        scoring='average_precision', # Optimize for Average Precision
        cv=3, # validation croisée à 3 plis
        verbose=1, # affiche la progression for GridSearchCV
        n_jobs=-1 # tous les cœurs disponibles
    )

    grid_search_catboost.fit(X_train_op03, y_train_op03)
    print("GridSearchCV for CatBoost completed.")

    # Meilleurs paramètres trouvés par GridSearchCV
    print(f"Best CatBoost parameters found: {grid_search_catboost.best_params_}")

    # Meilleur score
    print(f"Best CatBoost Average Precision score (CV): {grid_search_catboost.best_score_:.4f}")

    # Meilleur estimateur de la recherche
    best_cat_model = grid_search_catboost.best_estimator_

    # Prédiction sur le test avec le meilleur modèle
    y_pred_catboost = best_cat_model.predict(X_test_op03)
    y_pred_proba_catboost = best_cat_model.predict_proba(X_test_op03)[:, 1]

    # Évaluation du meilleur modèle CatBoost
    accuracy_catboost = accuracy_score(y_test_op03, y_pred_catboost)
    precision_catboost = precision_score(y_test_op03, y_pred_catboost)
    recall_catboost = recall_score(y_test_op03, y_pred_catboost)
    roc_auc_catboost = roc_auc_score(y_test_op03, y_pred_proba_catboost)
    average_precision_catboost = average_precision_score(y_test_op03, y_pred_proba_catboost)

    print(f"\n--- Tuned CatBoost Model Performance on op_03 Test Set ---")
    print(f"Accuracy: {accuracy_catboost:.4f}")
    print(f"Precision: {precision_catboost:.4f}")
    print(f"Recall: {recall_catboost:.4f}")
    print(f"ROC AUC: {roc_auc_catboost:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision_catboost:.4f}")

    # Matrice de confusion du meilleur modèle CatBoost
    cm_catboost = confusion_matrix(y_test_op03, y_pred_catboost)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_catboost, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for Tuned CatBoost Fraud Prediction (op_03)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### Importance des features — CatBoost

Mêmes importances côté CatBoost, pour comparer la hiérarchie des features d'un modèle à l'autre.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'best_cat_model' not in locals() and 'grid_search_catboost' in locals():
    best_cat_model = grid_search_catboost.best_estimator_
elif 'best_cat_model' not in locals() and 'grid_search_catboost' not in locals():
    print("Error: 'best_cat_model' and 'grid_search_catboost' are not defined. Please run the CatBoost model training and GridSearchCV cells first.")

if 'X_train_op03' not in locals():
    print("Erreur : X_train_op03 non défini. Exécuter d'abord la cellule de séparation train / test pour CatBoost.")

# Importance des features du meilleur modèle CatBoost
if 'best_cat_model' in locals() and 'X_train_op03' in locals():
    feature_importances_cat = best_cat_model.get_feature_importance()
    feature_names_cat = X_train_op03.columns

    # Mise en DataFrame pour l'affichage
    importance_df_cat = pd.DataFrame({
        'Feature': feature_names_cat,
        'Importance': feature_importances_cat
    })

    # Tri par importance
    importance_df_cat = importance_df_cat.sort_values(by='Importance', ascending=False)

    # Importance des features
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_cat, palette='plasma')
    plt.title('CatBoost Feature Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot CatBoost feature importance: Missing required variables.")

# Echec

In [ ]:
import datetime

# --- Features nécessaires ---

# Comptes classés
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

all_stripped_accounts = pd.concat([
    df['origin_account_stripped'],
    df['destination_account_stripped']
]).unique()

sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

df['origin_account_ranked'] = df['origin_account_stripped'].map(account_to_rank_mapping)
df['destination_account_ranked'] = df['destination_account_stripped'].map(account_to_rank_mapping)

# Conversions décimale, ipv4, mac, horodatage
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

df['origin_hex_clean'] = df['origin_account'].apply(clean_hex)
df['destination_hex_clean'] = df['destination_account'].apply(clean_hex)

df['calc_origin_decimal']   = df['origin_hex_clean'].apply(hex_to_decimal)
df['calc_origin_ipv4']      = df['origin_hex_clean'].apply(hex_to_ipv4)
df['calc_origin_mac']       = df['origin_hex_clean'].apply(hex_to_mac)
df['calc_origin_timestamp'] = df['origin_hex_clean'].apply(hex_to_timestamp)

df['calc_destination_decimal']   = df['destination_hex_clean'].apply(hex_to_decimal)
df['calc_destination_ipv4']      = df['destination_hex_clean'].apply(hex_to_ipv4)
df['calc_destination_mac']       = df['destination_hex_clean'].apply(hex_to_mac)
df['calc_destination_timestamp'] = df['destination_hex_clean'].apply(hex_to_timestamp)

df.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')


# Drapeaux has_timestamp
df['has_origin_timestamp'] = df['calc_origin_timestamp'].notna()
df['has_destination_timestamp'] = df['calc_destination_timestamp'].notna()


# Drapeaux previously_fraud
# Comptes émetteurs impliqués dans une fraude
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
# Comptes destinataires impliqués dans une fraude
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
# Ensemble des comptes ayant fraudé au moins une fois (émetteur ou destinataire)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
# Drapeaux d'historique de fraude pour l'émetteur et le destinataire
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# --- Fin des features ---

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

df = df[df['operation']=='op_03'].copy()

# Cible
y_previously_fraud = df['destination_account_previously_fraud']

# Features pour prédire destination_account_previously_fraud
# Hors fraud_flag, la cible et les identifiants directs
features_for_prev_fraud = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

# Présence des features et NaN traités avant la séparation
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

X_previously_fraud = df[features_for_prev_fraud]

# Suppression des lignes avec NaN résiduels dans les features
X_previously_fraud = X_previously_fraud.dropna()
y_previously_fraud = y_previously_fraud[X_previously_fraud.index]

if X_previously_fraud.empty:
    print("After data cleaning, no entries remain for training the 'previously fraudulent' model.")
elif len(y_previously_fraud.unique()) < 2: # vérifie la variabilité de la cible
    print(f"Only one class present in the target variable ('destination_account_previously_fraud'). Cannot train a classifier.")
else:
    # Séparation train / test
    X_train_pf, X_test_pf, y_train_pf, y_test_pf = train_test_split(X_previously_fraud, y_previously_fraud, test_size=0.3, random_state=42, stratify=y_previously_fraud)

    # Entraînement XGBoost
    model_previously_fraud = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
    model_previously_fraud.fit(X_train_pf, y_train_pf)

    # Évaluation
    y_pred_pf = model_previously_fraud.predict(X_test_pf)
    y_pred_proba_pf = model_previously_fraud.predict_proba(X_test_pf)[:, 1]

    accuracy_pf = accuracy_score(y_test_pf, y_pred_pf)
    precision_pf = precision_score(y_test_pf, y_pred_pf)
    recall_pf = recall_score(y_test_pf, y_pred_pf)
    roc_auc_pf = roc_auc_score(y_test_pf, y_pred_proba_pf)

    print(f"\n--- Model to Predict Previously Fraudulent Destination Accounts ---")
    print(f"Accuracy: {accuracy_pf:.4f}")
    print(f"Precision: {precision_pf:.4f}")
    print(f"Recall: {recall_pf:.4f}")
    print(f"ROC AUC: {roc_auc_pf:.4f}")

    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    cm_pf = confusion_matrix(y_test_pf, y_pred_pf)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_pf, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Previously Fraud (0)', 'Previously Fraud (1)'],
                yticklabels=['Actual Not Previously Fraud (0)', 'Actual Previously Fraud (1)'])
    plt.title('Confusion Matrix for Previously Fraudulent Destination Account Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Étage 1 : prédire destination_account_previously_fraud sur tout le jeu ---
print("\n--- Stage 1: Predicting 'destination_account_previously_fraud' ---")

df_stage1 = df.copy()

# Features du premier modèle
if 'features_for_prev_fraud' not in locals():
    features_for_prev_fraud = [
        'period',
        'amount',
        'origin_balance_before',
        'origin_balance_after',
        'destination_balance_before',
        'destination_balance_after',
        'origin_account_ranked',
        'destination_account_ranked',
        'has_origin_timestamp',
        'has_destination_timestamp',
        'origin_account_previously_fraud',
        'calc_origin_decimal',
        'calc_destination_decimal'
    ]

X_full_for_prev_fraud_prediction = df_stage1[features_for_prev_fraud]

# Prédiction de destination_account_previously_fraud
if 'model_previously_fraud' not in locals():
    print("Erreur : model_previously_fraud introuvable. Exécuter d'abord la cellule qui l'entraîne.")
else:
    predicted_dest_prev_fraud = model_previously_fraud.predict(X_full_for_prev_fraud_prediction)
    df_stage1['predicted_destination_account_previously_fraud'] = predicted_dest_prev_fraud

    # Évaluation de l'étage 1 contre le vrai destination_account_previously_fraud
    y_true_dest_prev_fraud = df_stage1['destination_account_previously_fraud']
    accuracy_s1 = accuracy_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    precision_s1 = precision_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    recall_s1 = recall_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    roc_auc_s1 = roc_auc_score(y_true_dest_prev_fraud, model_previously_fraud.predict_proba(X_full_for_prev_fraud_prediction)[:, 1])

    print(f"Stage 1 Metrics (predicting destination_account_previously_fraud):")
    print(f"  Accuracy: {accuracy_s1:.4f}")
    print(f"  Precision: {precision_s1:.4f}")
    print(f"  Recall: {recall_s1:.4f}")
    print(f"  ROC AUC: {roc_auc_s1:.4f}")

    # Colonnes de prédiction hiérarchique
    df_stage1['hierarchical_predicted_fraud_flag'] = 0 # prédictions dures
    df_stage1['fraud_probability'] = 0.0 # prédictions souples, pour la soumission

    # --- Étage 2 : modèle fraud_flag sur le sous-ensemble filtré ---
    print("\n--- Stage 2: Training model for 'fraud_flag' on susceptible op_03 transactions ---")

    # Transactions op_03 dont l'étage 1 prédit un destinataire déjà frauduleux
    # On encode ici le fait que la fraude n'existe que dans op_03
    df_susceptible_op03_fraud = df_stage1[
        (df_stage1['predicted_destination_account_previously_fraud'] == 1)
    ].copy()

    if df_susceptible_op03_fraud.empty:
        print("No 'op_03' transactions predicted as susceptible to fraud. Stage 2 model will not be trained.")
        # fraud_probability et hierarchical_predicted_fraud_flag restent à 0 ailleurs, ce qui est correct
    else:
        print(f"Number of 'op_03' transactions susceptible to fraud: {len(df_susceptible_op03_fraud)}")

        # Features de l'étage 2, identiques à features_for_prev_fraud
        features_for_stage2_model = [f for f in features_for_prev_fraud if f != 'destination_account_previously_fraud'] # Remove if it was mistakenly included before

        X_stage2 = df_susceptible_op03_fraud[features_for_stage2_model]
        y_stage2 = df_susceptible_op03_fraud['fraud_flag']

        if X_stage2.empty or len(y_stage2.unique()) < 2:
            print("Insufficient data or only one class in susceptible 'op_03' transactions for Stage 2 model. All fraud probabilities for these will be 0.")
        else:
            X_train_s2, X_test_s2, y_train_s2, y_test_s2 = train_test_split(
                X_stage2, y_stage2, test_size=0.3, random_state=42, stratify=y_stage2
            )

            model_fraud_flag_stage2 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
            model_fraud_flag_stage2.fit(X_train_s2, y_train_s2)

            y_pred_s2 = model_fraud_flag_stage2.predict(X_test_s2)
            y_pred_proba_s2 = model_fraud_flag_stage2.predict_proba(X_test_s2)[:, 1]

            accuracy_s2 = accuracy_score(y_test_s2, y_pred_s2)
            precision_s2 = precision_score(y_test_s2, y_pred_s2)
            recall_s2 = recall_score(y_test_s2, y_pred_s2)
            roc_auc_s2 = roc_auc_score(y_test_s2, y_pred_proba_s2)

            print(f"Stage 2 Metrics (predicting fraud_flag on susceptible 'op_03' subset):")
            print(f"  Accuracy: {accuracy_s2:.4f}")
            print(f"  Precision: {precision_s2:.4f}")
            print(f"  Recall: {recall_s2:.4f}")
            print(f"  ROC AUC: {roc_auc_s2:.4f}")

            # Probabilités de fraude sur tout le sous-ensemble op_03 susceptible
            predicted_fraud_proba_for_susceptible_op03 = model_fraud_flag_stage2.predict_proba(X_stage2)[:, 1]
            df_stage1.loc[df_susceptible_op03_fraud.index, 'fraud_probability'] = predicted_fraud_proba_for_susceptible_op03

            # Mise à jour de hierarchical_predicted_fraud_flag sur ce sous-ensemble
            # (prédictions dures de l'étage 2)
            predicted_fraud_flag_for_susceptible_op03 = model_fraud_flag_stage2.predict(X_stage2)
            df_stage1.loc[df_susceptible_op03_fraud.index, 'hierarchical_predicted_fraud_flag'] = predicted_fraud_flag_for_susceptible_op03

    # --- Métriques globales du modèle hiérarchique ---
    print("\n--- Global Metrics for the Hierarchical Prediction Model ---")
    y_true_global = df_stage1['fraud_flag']
    y_pred_global = df_stage1['hierarchical_predicted_fraud_flag']
    y_proba_global = df_stage1['fraud_probability']

    global_accuracy = accuracy_score(y_true_global, y_pred_global)

    # Division par zéro de precision_score si aucune prédiction positive
    if (y_pred_global == 1).sum() > 0:
        global_precision = precision_score(y_true_global, y_pred_global)
    else:
        global_precision = 0.0 # précision nulle si aucune fraude prédite

    global_recall = recall_score(y_true_global, y_pred_global)
    global_roc_auc = roc_auc_score(y_true_global, y_proba_global)


    print(f"Global Accuracy: {global_accuracy:.4f}")
    print(f"Global Precision: {global_precision:.4f}")
    print(f"Global Recall: {global_recall:.4f}")
    print(f"Global ROC AUC (using probabilities): {global_roc_auc:.4f}")

    # Matrice de confusion, modèle global
    cm_global = confusion_matrix(y_true_global, y_pred_global)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_global, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Global Confusion Matrix for Hierarchical Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### sumision

In [ ]:
test = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')

In [ ]:
import pandas as pd
import datetime

# --- Fonctions de feature engineering ---
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

if 'test' not in locals():
    print("Error: 'test' DataFrame not found. Please ensure test.csv is loaded.")

# Copie pour ne pas modifier test
test_submission = test.copy()

# --- Features du test, alignées sur le traitement du train ---

# 1. Nettoyage et classement des comptes
if 'account_to_rank_mapping' not in locals() or 'df' not in locals():
    print("Warning: 'account_to_rank_mapping' or 'df' not found. Regenerating from df...")
    df_temp_for_mapping = df.copy() # copie, pour ne pas modifier df
    df_temp_for_mapping['origin_account_stripped'] = df_temp_for_mapping['origin_account'].str.replace('acc_o_', '')
    df_temp_for_mapping['destination_account_stripped'] = df_temp_for_mapping['destination_account'].str.replace('acc_d_', '')
    all_stripped_accounts = pd.concat([
        df_temp_for_mapping['origin_account_stripped'],
        df_temp_for_mapping['destination_account_stripped']
    ]).unique()
    sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)
    account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

test_submission['origin_account_stripped'] = test_submission['origin_account'].str.replace('acc_o_', '')
test_submission['destination_account_stripped'] = test_submission['destination_account'].str.replace('acc_d_', '')

test_submission['origin_account_ranked'] = test_submission['origin_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)
test_submission['destination_account_ranked'] = test_submission['destination_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)

# 2. Conversions hexadécimal vers décimal, ipv4, mac, horodatage
test_submission['origin_hex_clean'] = test_submission['origin_account'].apply(clean_hex)
test_submission['destination_hex_clean'] = test_submission['destination_account'].apply(clean_hex)

test_submission['calc_origin_decimal']   = test_submission['origin_hex_clean'].apply(hex_to_decimal)
test_submission['calc_origin_ipv4']      = test_submission['origin_hex_clean'].apply(hex_to_ipv4)
test_submission['calc_origin_mac']       = test_submission['origin_hex_clean'].apply(hex_to_mac)
test_submission['calc_origin_timestamp'] = test_submission['origin_hex_clean'].apply(hex_to_timestamp)

test_submission['calc_destination_decimal']   = test_submission['destination_hex_clean'].apply(hex_to_decimal)
test_submission['calc_destination_ipv4']      = test_submission['destination_hex_clean'].apply(hex_to_ipv4)
test_submission['calc_destination_mac']       = test_submission['destination_hex_clean'].apply(hex_to_mac)
test_submission['calc_destination_timestamp'] = test_submission['destination_hex_clean'].apply(hex_to_timestamp)

test_submission.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')

# 3. Drapeaux has_timestamp
test_submission['has_origin_timestamp'] = test_submission['calc_origin_timestamp'].notna()
test_submission['has_destination_timestamp'] = test_submission['calc_destination_timestamp'].notna()

# 4. Drapeaux previously_fraud
if 'fraudulent_origin_accounts' not in locals() or 'fraudulent_destination_accounts' not in locals() or 'df' not in locals():
    print("Warning: Fraudulent account lists or 'df' not found. Regenerating from df...")
    fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

test_submission['origin_account_previously_fraud'] = test_submission['origin_account'].isin(all_fraudulent_accounts).astype(int)
test_submission['destination_account_previously_fraud'] = test_submission['destination_account'].isin(all_fraudulent_accounts).astype(int)

# --- Features des modèles, identiques au train ---
features_for_prev_fraud = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

features_for_stage2_model = [f for f in features_for_prev_fraud if f != 'destination_account_previously_fraud']

# NaN de calc_origin_decimal et calc_destination_decimal sur le test
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in test_submission.columns:
        test_submission[col] = test_submission[col].fillna(0)

# --- Étage 1 : prédiction de destination_account_previously_fraud ---
if 'model_previously_fraud' not in locals():
    print("Error: 'model_previously_fraud' not found. Please train Stage 1 model first.")
else:
    X_test_stage1 = test_submission[features_for_prev_fraud]
    test_submission['predicted_destination_account_previously_fraud'] = model_previously_fraud.predict(X_test_stage1)
    test_submission['fraud_probability'] = 0.0 # probabilités initialisées à 0

    # --- Étage 2 : prédiction de fraud_flag sur les transactions susceptibles ---
    # Transactions susceptibles selon l'étage 1
    susceptible_transactions_test = test_submission[test_submission['predicted_destination_account_previously_fraud'] == 1].copy()

    if not susceptible_transactions_test.empty:
        if 'model_fraud_flag_stage2' not in locals():
            print("Error: 'model_fraud_flag_stage2' not found. Please train Stage 2 model first.")
        else:
            X_test_stage2 = susceptible_transactions_test[features_for_stage2_model]

            # Probabilités de fraud_flag sur ce sous-ensemble
            predicted_fraud_proba_stage2 = model_fraud_flag_stage2.predict_proba(X_test_stage2)[:, 1]

            # Mise à jour de fraud_probability dans test_submission
            test_submission.loc[susceptible_transactions_test.index, 'fraud_probability'] = predicted_fraud_proba_stage2
    else:
        print("No transactions predicted as susceptible to fraud by Stage 1 model. All target probabilities will be 0.")

    # --- Fichier de soumission ---
    submission = test_submission[['id', 'fraud_probability']].rename(columns={'fraud_probability': 'target'})

    # Écriture de la soumission
    submission.to_csv('submission.csv', index=False)

    print("Submission file 'submission.csv' created successfully using the hierarchical model.")
    print(submission.head())

## FAST TESTING 02

### Matrice de confusion

Détail des vrais et faux positifs et négatifs, pour regarder de près les faux négatifs.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Visualisation de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-Fraude Prédit (0)', 'Fraude Prédite (1)'],
            yticklabels=['Non-Fraude Réelle (0)', 'Fraude Réelle (1)'])
plt.title('Matrice de Confusion pour le Modèle XGBoost')
plt.xlabel('Prédiction')
plt.ylabel('Valeur Réelle')
plt.show()

# Extraction des valeurs pour une analyse plus facile
true_negatives = cm[0, 0]
false_positives = cm[0, 1]
false_negatives = cm[1, 0]
true_positives = cm[1, 1]

print(f"\nVrais Négatifs (True Negatives - TN): {true_negatives}")
print(f"Faux Positifs (False Positives - FP): {false_positives}")
print(f"Faux Négatifs (False Negatives - FN): {false_negatives}")
print(f"Vrais Positifs (True Positives - TP): {true_positives}")

print("\nAnalyse des Faux Négatifs (FN): Ce sont les transactions qui étaient réellement frauduleuses (Fraud Flag = 1) mais que le modèle a prédites comme non-frauduleuses (Fraud Flag = 0). Un nombre élevé de faux négatifs indique que le modèle manque de nombreuses fraudes réelles. C'est souvent un point critique dans la détection de fraude.")
print("Analyse des Faux Positifs (FP): Ce sont les transactions qui étaient non-frauduleuses (Fraud Flag = 0) mais que le modèle a prédites comme frauduleuses (Fraud Flag = 1). Un nombre élevé de faux positifs peut entraîner des investigations inutiles ou un rejet erroné de transactions légitimes.")

## EDA 02

In [ ]:
df.columns

In [ ]:
unique_op03_destination_accounts = df[df['operation'] == 'op_03']['destination_account'].unique()

accounts_with_other_operations = []

for account in unique_op03_destination_accounts:
    # Opérations du compte destinataire courant
    operations_for_account = df[df['destination_account'] == account]['operation'].unique()

    # Présence d'opérations autres que op_03
    other_operations = [op for op in operations_for_account if op != 'op_03']

    if other_operations:
        accounts_with_other_operations.append({
            'destination_account': account,
            'other_operations': other_operations
        })

if accounts_with_other_operations:
    print(f"Number of destination accounts involved in 'op_03' and other operations: {len(accounts_with_other_operations)}")
    print("Details for some of these accounts:")
    for i, acc_info in enumerate(accounts_with_other_operations[:5]): # les 5 premiers
        print(f"  Account: {acc_info['destination_account']}, Other Operations: {acc_info['other_operations']}")
else:
    print("No destination accounts involved in 'op_03' are also involved in other operations.")

In [ ]:
df.head()

In [ ]:
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

# Union des ID de comptes nettoyés, émetteurs et destinataires
all_stripped_accounts = pd.concat([
    df['origin_account_stripped'],
    df['destination_account_stripped']
]).unique()

# Tri alphabétique des comptes distincts
sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

# Association compte -> rang numérique (à partir de 1)
account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

# Colonnes de rang dans df
df['origin_account_ranked'] = df['origin_account_stripped'].map(account_to_rank_mapping)
df['destination_account_ranked'] = df['destination_account_stripped'].map(account_to_rank_mapping)

print("New columns 'origin_account_ranked' and 'destination_account_ranked' created.")
print("First 5 rows with new ranked account IDs:")
print(df[['origin_account', 'origin_account_stripped', 'origin_account_ranked',
          'destination_account', 'destination_account_stripped', 'destination_account_ranked']].head())

In [ ]:
import pandas as pd

df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')

# Étape 1 : récapitulatif des comptes destinataires uniques
destination_summary = df.groupby('destination_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('destination_account', 'size')
).reset_index()

# Ajout de l'ID de compte nettoyé au récapitulatif
account_stripped_mapping = df[['destination_account', 'destination_account_stripped']].drop_duplicates()
destination_summary = pd.merge(destination_summary, account_stripped_mapping, on='destination_account', how='left')

print("Résumé des comptes destinataires uniques:")
print(destination_summary.head())

# Récapitulatif des comptes émetteurs uniques
origin_summary = df.groupby('origin_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('origin_account', 'size')
).reset_index()

# Ajout de l'ID de compte nettoyé au récapitulatif
origin_stripped_mapping = df[['origin_account', 'origin_account_stripped']].drop_duplicates()
origin_summary = pd.merge(origin_summary, origin_stripped_mapping, on='origin_account', how='left')

print("\nRésumé des comptes émetteurs uniques:")
print(origin_summary.head())

# Extraction des segments
def extract_hex_part(hex_string, start, length):
    if pd.isna(hex_string) or not isinstance(hex_string, str) or len(hex_string) < start + length:
        return None
    return hex_string[start:start+length]

# Étape 2 : segments hexadécimaux ajoutés à destination_summary
# Découpage en segments de 2 caractères (hex de 16 caractères)
for i in range(8):
    start_index = i * 2
    col_name = f'dest_hex_p{i+1}_2'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Découpage en segments de 4 caractères (hex de 16 caractères)
for i in range(4):
    start_index = i * 4
    col_name = f'dest_hex_p{i+1}_4'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes destinataires avec les parties hexadécimales extraites:")
print(destination_summary.head())

# Segments hexadécimaux ajoutés à origin_summary
# Découpage en segments de 2 caractères (hex de 16 caractères)
for i in range(8):
    start_index = i * 2
    col_name = f'origin_hex_p{i+1}_2'
    origin_summary[col_name] = origin_summary['origin_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Découpage en segments de 4 caractères (hex de 16 caractères)
for i in range(4):
    start_index = i * 4
    col_name = f'origin_hex_p{i+1}_4'
    origin_summary[col_name] = origin_summary['origin_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes émetteurs avec les parties hexadécimales extraites:")
print(origin_summary.head())

# Étape 3 : répétitions côté comptes destinataires
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées (Destinataires) --")

dest_hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

for col in dest_hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Comptage des valeurs, hors None
    value_counts = destination_summary[col].value_counts(dropna=True)
    # Valeurs apparaissant plus d'une fois
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

# Répétitions côté comptes émetteurs
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées (Émetteurs) --")

origin_hex_part_columns = [col for col in origin_summary.columns if 'origin_hex_p' in col]

for col in origin_hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Comptage des valeurs, hors None
    value_counts = origin_summary[col].value_counts(dropna=True)
    # Valeurs apparaissant plus d'une fois
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

# Étape 4 : comparaison des segments hexadécimaux émetteur / destinataire
print("\n-- Comparaison des Parties Hexadécimales Communes (Émetteurs vs. Destinataires) --")

common_hex_segments_details = []

for dest_col in dest_hex_part_columns:
    # Colonne émetteur correspondante
    # Exemple : dest_hex_p1_2 -> origin_hex_p1_2
    origin_col = dest_col.replace('dest_hex_p', 'origin_hex_p')

    if origin_col in origin_hex_part_columns:
        dest_unique_segments = set(destination_summary[dest_col].dropna().unique())
        origin_unique_segments = set(origin_summary[origin_col].dropna().unique())

        common_segments = dest_unique_segments.intersection(origin_unique_segments)

        if common_segments:
            common_hex_segments_details.append({
                'segment_type': dest_col.split('_')[1] + '_' + dest_col.split('_')[2], # par exemple p1_2
                'count': len(common_segments),
                'examples': list(common_segments)[:5] # jusqu'à 5 exemples
            })

if common_hex_segments_details:
    total_common_segments = sum([item['count'] for item in common_hex_segments_details])
    print(f"Au total, il y a {total_common_segments} segments hexadécimaux uniques qui apparaissent à la fois dans les comptes émetteurs et récepteurs.")
    print("Détails des segments communs par type de segment :")
    for detail in common_hex_segments_details:
        print(f"  - Type de segment ({detail['segment_type']}): {detail['count']} segments communs, exemples : {detail['examples']}")
else:
    print("Aucun segment hexadécimal commun n'a été trouvé entre les comptes émetteurs et récepteurs.")

In [ ]:
import pandas as pd
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

# Étape 1 : récapitulatif des comptes destinataires uniques
destination_summary = df.groupby('destination_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('destination_account', 'size')
).reset_index()

# Ajout de l'ID de compte nettoyé au récapitulatif
# destination_account_stripped est déjà dans df
# Fusion sur destination_account
account_stripped_mapping = df[['destination_account', 'destination_account_stripped']].drop_duplicates()
destination_summary = pd.merge(destination_summary, account_stripped_mapping, on='destination_account', how='left')

print("Résumé des comptes destinataires uniques:")
print(destination_summary.head())

# Étape 2 : segments hexadécimaux ajoutés à destination_summary
# Extraction des segments
def extract_hex_part(hex_string, start, length):
    if pd.isna(hex_string) or not isinstance(hex_string, str) or len(hex_string) < start + length:
        return None
    return hex_string[start:start+length]

# Découpage en segments de 2 caractères (hex de 16 caractères)
for i in range(8):
    start_index = i * 2
    col_name = f'dest_hex_p{i+1}_2'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Découpage en segments de 4 caractères (hex de 16 caractères)
for i in range(4):
    start_index = i * 4
    col_name = f'dest_hex_p{i+1}_4'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes destinataires avec les parties hexadécimales extraites:")
print(destination_summary.head())

# Étape 3 : répétitions
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées --")

hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

for col in hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Comptage des valeurs, hors None
    value_counts = destination_summary[col].value_counts(dropna=True)
    # Valeurs apparaissant plus d'une fois
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

### Corrélation entre segments hexadécimaux et fraude

Taux de fraude par segment hexadécimal extrait, et repérage des segments les plus et les moins associés à la fraude.

In [ ]:
hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

correlation_results = {}

for col in hex_part_columns:
    # Somme de fraud_count et total_occurrences par segment
    grouped_data = destination_summary.groupby(col).agg(
        total_fraud_for_segment=('fraud_count', 'sum'),
        total_transactions_for_segment=('total_occurrences', 'sum')
    ).reset_index()

    # Taux de fraude par segment
    # Division par zéro si total_transactions_for_segment vaut 0
    grouped_data['fraud_rate'] = grouped_data.apply(lambda row: row['total_fraud_for_segment'] / row['total_transactions_for_segment'] if row['total_transactions_for_segment'] > 0 else 0, axis=1)

    # Résultats triés par fraud_rate
    correlation_results[col] = grouped_data.sort_values(by='fraud_rate', ascending=False)

# Extrêmes haut et bas par colonne
for col, result_df in correlation_results.items():
    print(f"\n--- Top 5 Hex Segments with Highest Fraud Rate for {col} ---")
    # Hors NaN, et uniquement les segments avec des transactions
    display(result_df[result_df['total_transactions_for_segment'] > 0].head(5))

    # Quelques segments à taux de fraude nul, pour contraste
    print(f"--- Top 5 Hex Segments with Lowest (or Zero) Fraud Rate for {col} ---")
    display(result_df[result_df['total_transactions_for_segment'] > 0].sort_values(by='fraud_rate', ascending=True).head(5))

### Visualisation de la distribution des taux de fraude par segment

In [ ]:
for col, result_df in correlation_results.items():
    plt.figure(figsize=(10, 6))
    # Hors segments sans transaction (taux de fraude nul mais non significatif)
    # et hors segments None
    meaningful_fraud_rates = result_df[result_df['total_transactions_for_segment'] > 0]['fraud_rate'].dropna()

    if not meaningful_fraud_rates.empty:
        sns.histplot(meaningful_fraud_rates, bins=30, kde=True)
        plt.title(f'Distribution du Taux de Fraude pour le Segment {col}')
        plt.xlabel('Taux de Fraude')
        plt.ylabel('Fréquence')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.show()
    else:
        print(f"Aucune donnée significative pour la visualisation du taux de fraude pour le segment {col}")

### Test du chi-deux d'indépendance

Confirmation statistique de l'association entre segments hexadécimaux et fraude.

**Variables croisées :**
1. Le segment hexadécimal (`dest_hex_p1_2` par exemple), catégorielle à nombreuses modalités.
2. `fraud_flag`, binaire (0 = non-fraude, 1 = fraude).

**Hypothèses :**
- **H0** : pas d'association — le taux de fraude est le même pour tous les segments.
- **H1** : association — les taux de fraude diffèrent significativement entre segments.

**Lecture de la p-value :**
- **p < 0.05** : H0 rejetée, association significative entre le segment et la fraude.
- **p ≥ 0.05** : H0 conservée, pas de preuve d'association.

### Segments hexadécimaux aux taux de fraude les plus élevés

Une fois l'association confirmée, on isole les segments qui concentrent le plus de fraude.

In [ ]:
print("\n--- Top 5 Hex Segments with Highest Fraud Rate Across All Columns ---")

for col, result_df in correlation_results.items():
    # Hors NaN, et uniquement les segments avec des transactions
    top_fraud_segments = result_df[result_df['total_transactions_for_segment'] > 0].head(5)
    if not top_fraud_segments.empty:
        print(f"\nTop 5 for {col}:")
        display(top_fraud_segments[[col, 'fraud_rate']].rename(columns={col: 'Segment'}))
    else:
        print(f"\nNo top fraud segments to display for {col}.")

In [ ]:
from scipy.stats import chi2_contingency

print("--- Résultats du Test du Chi-Carré pour l'Association Hexadécimal-Fraude ---")

for col, result_df in correlation_results.items():
    # Hors segments sans transaction ou valant None,
    # pour un test statistique significatif
    meaningful_data = result_df[result_df['total_transactions_for_segment'] > 0].dropna(subset=[col])

    if not meaningful_data.empty:
        # Transactions non frauduleuses par segment
        meaningful_data['total_non_fraud_for_segment'] = meaningful_data['total_transactions_for_segment'] - meaningful_data['total_fraud_for_segment']

        # Table de contingence : effectifs par segment, fraude contre non-fraude
        # chi2_contingency attend un tableau 2D
        contingency_table = meaningful_data[['total_fraud_for_segment', 'total_non_fraud_for_segment']].values

        # Test du chi-deux
        chi2, p_value, _, _ = chi2_contingency(contingency_table)

        print(f"\nSegment: {col}")
        print(f"  Valeur Chi-Carré : {chi2:.2f}")
        print(f"  P-value : {p_value:.4f}")

        if p_value < 0.05:
            print("  Conclusion : Rejet de l'hypothèse nulle. Il existe une association significative entre ce segment hexadécimal et le `fraud_flag`.")
        else:
            print("  Conclusion : Incapacité à rejeter l'hypothèse nulle. Pas de preuve d'une association significative entre ce segment hexadécimal et le `fraud_flag`.")
    else:
        print(f"\nSegment: {col}")
        print("  Pas de données suffisantes pour effectuer le test du Chi-Carré.")

## EDAAAA

In [ ]:
df.columns

In [ ]:
df.head(10)

### adresse et date

In [ ]:
def extract_ipv4_network(ipv4_address, prefix_length=24):
    if ipv4_address is None: # gestion des valeurs None
        return None
    try:
        # Découpage de l'adresse IP en octets
        octets = ipv4_address.split('.')
        if len(octets) != 4:
            return None

        # Nombre d'octets de la partie réseau
        num_octets = prefix_length // 8
        network_part = octets[:num_octets]

        # Si prefix_length n'est pas un multiple de 8, le dernier octet est partiel
        if prefix_length % 8 != 0:
            # On s'en tient aux frontières d'octets
            # Pour /24 : les trois premiers octets
            # Pour /16 : les deux premiers octets
            pass # gère /8, /16, /24 et /32

        return ".".join(network_part)
    except Exception:
        return None

# Colonnes de réseau IPv4 émetteur et destinataire (/24)
df['origin_ipv4_network_24'] = df['origin_ipv4'].apply(lambda x: extract_ipv4_network(x, 24))
df['destination_ipv4_network_24'] = df['destination_ipv4'].apply(lambda x: extract_ipv4_network(x, 24))

print("Analyse des réseaux IPv4 des comptes émetteurs:")
origin_network_analysis = df.groupby(['origin_ipv4_network_24', 'fraud_flag']).size().unstack(fill_value=0)
print(origin_network_analysis.head(10))

print("\nAnalyse des réseaux IPv4 des comptes récepteurs:")
destination_network_analysis = df.groupby(['destination_ipv4_network_24', 'fraud_flag']).size().unstack(fill_value=0)
print(destination_network_analysis.head(10))

# Principaux réseaux, fraude et non-fraude
print("\nTop 10 réseaux d'origine avec le plus de fraudes (Flag=1):")
print(origin_network_analysis.sort_values(by=1, ascending=False).head(10))

print("\nTop 10 réseaux de destination avec le plus de fraudes (Flag=1):")
print(destination_network_analysis.sort_values(by=1, ascending=False).head(10))

print("\nTop 10 réseaux d'origine avec le plus de non-fraudes (Flag=0):")
print(origin_network_analysis.sort_values(by=0, ascending=False).head(10))

print("\nTop 10 réseaux de destination avec le plus de non-fraudes (Flag=0):")
print(destination_network_analysis.sort_values(by=0, ascending=False).head(10))

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 1. CRÉATION DU GRAPHE INFORMATIQUE (DIRIGÉ)
# Les nœuds représentent les machines (IP / MAC) et les arêtes le flux réseau.
G_network = nx.DiGraph()

# 2. CONSTRUCTION DU RÉSEAU À PARTIR DES IPS ET MACS
# Colonnes origin_ipv4, origin_mac, etc.
for _, row in df.iterrows():
    # On ignore les lignes sans adresse IP ou MAC (NaT ou NaN)
    if pd.isna(row['origin_ipv4']) or pd.isna(row['destination_ipv4']):
        continue

    src_ip = row['origin_ipv4']
    dst_ip = row['destination_ipv4']
    src_mac = row['origin_mac']
    dst_mac = row['destination_mac']

    # Ajout des machines (nœuds) avec leurs attributs matériels
    G_network.add_node(src_ip, mac=src_mac, type="Émetteur")
    G_network.add_node(dst_ip, mac=dst_mac, type="Récepteur")

    # Ajout du lien réseau (flux de paquets) avec le volume de données (amount) et l'opération
    G_network.add_edge(
        src_ip,
        dst_ip,
        volume_data=float(row['amount']),
        protocol=row['operation']
    )

print("--- ANALYSE DE LA TOPOLOGIE DU RÉSEAU INFORMATIQUE ---")
print(f"Nombre de machines actives détectées (IPs) : {G_network.number_of_nodes()}")
print(f"Nombre de connexions / flux réseau actifs  : {G_network.number_of_edges()}\n")


# 3. ANALYSE DU TRAFIC (MÉTRIQUES RÉSEAU)

# Trafic Sortant (Machines qui émettent le plus de flux)
out_flux = dict(G_network.out_degree())
# Trafic Entrant (Machines serveurs ou cibles qui reçoivent le plus de connexions)
in_flux = dict(G_network.in_degree())

# Détection des Routeurs / Hubs Centraux (Betweenness Centrality)
# Identifie les machines par lesquelles passent la majorité des paquets du réseau
hubs_reseau = nx.betweenness_centrality(G_network)


# 4. CRÉATION DU RAPPORT D'AUDIT DU RÉSEAU
infrastructure_report = pd.DataFrame({
    'MAC_Adresse': pd.Series(nx.get_node_attributes(G_network, 'mac')),
    'Connexions_Sortantes': pd.Series(out_flux),
    'Connexions_Entrantes': pd.Series(in_flux),
    'Indice_Centralite_Hub': pd.Series(hubs_reseau)
}).fillna(0)

print("--- TOP 5 DES MACHINES AGISSANT COMME PASSERELLES / HUBS CENTRAUX ---")
print(infrastructure_report.sort_values(by='Indice_Centralite_Hub', ascending=False).head(5))
print("\n")


# 5. VISUALISATION ANTHROPOMORPHIQUE DU RÉSEAU
plt.figure(figsize=(14, 9))

# Algorithme de disposition (Layout) pour espacer proprement les sous-réseaux
pos = nx.kamada_kawai_layout(G_network)

# Définir la taille des nœuds selon leur importance dans le trafic total
node_sizes = [(in_flux[node] + out_flux[node]) * 300 + 100 for node in G_network.nodes()]

# Dessiner les équipements (Serveurs/Clients)
nx.draw_networkx_nodes(G_network, pos, node_size=node_sizes, node_color='springgreen', alpha=0.85)

# Dessiner les câbles réseau (Flèches directionnelles du trafic)
nx.draw_networkx_edges(G_network, pos, arrowstyle='->', arrowsize=12, edge_color='royalblue', width=1.2)

# Afficher les adresses IP sur la carte
nx.draw_networkx_labels(G_network, pos, font_size=8, font_family='sans-serif', font_weight='bold')

plt.title("Cartographie de la Topologie et des Flux du Réseau Informatique", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import datetime

# 1. Fonction de nettoyage et extraction de l'hexadécimal brut
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    # Supprime les préfixes courants 'acc_o_' ou 'acc_d_' et les espaces
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

# 2. Fonctions de conversion individuelles (gèrent les chaînes de 16 caractères hex)
def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            # Récupération des 8 derniers caractères pour l'IPv4
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            # Récupération des 12 derniers caractères pour l'adresse MAC
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            # Conversion basée sur l'hypothèse de nanosecondes Unix
            ts_sec = val_dec / 1_000_000_000
            # On limite aux dates valides gérées par Python
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

# --- APPLICATION SUR LE DATAFRAME ---

# Étape A: Nettoyage et isolation des chaînes hexadécimales brutes
df['origin_hex_clean'] = df['origin_account'].apply(clean_hex)
df['destination_hex_clean'] = df['destination_account'].apply(clean_hex)

# Étape B: Génération des colonnes de conversion pour l'ÉMETTEUR (Origin)
df['calc_origin_decimal']   = df['origin_hex_clean'].apply(hex_to_decimal)
df['calc_origin_ipv4']      = df['origin_hex_clean'].apply(hex_to_ipv4)
df['calc_origin_mac']       = df['origin_hex_clean'].apply(hex_to_mac)
df['calc_origin_timestamp'] = df['origin_hex_clean'].apply(hex_to_timestamp)

# Étape C: Génération des colonnes de conversion pour le RÉCEPTEUR (Destination)
df['calc_destination_decimal']   = df['destination_hex_clean'].apply(hex_to_decimal)
df['calc_destination_ipv4']      = df['destination_hex_clean'].apply(hex_to_ipv4)
df['calc_destination_mac']       = df['destination_hex_clean'].apply(hex_to_mac)
df['calc_destination_timestamp'] = df['destination_hex_clean'].apply(hex_to_timestamp)

# Optionnel: Supprimer les colonnes de travail temporaires si nécessaire
df.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True)

# Affichage des résultats pour vérification
df[['origin_account', 'calc_origin_ipv4', 'calc_origin_mac', 'calc_origin_timestamp']].head(3)


In [ ]:
def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16)
    except (ValueError, TypeError):
        return None

def hex_to_utf8(hex_str):
    try:
        # Longueur paire obligatoire pour bytes.fromhex
        if len(hex_str) % 2 != 0:
            hex_str = '0' + hex_str
        return bytes.fromhex(hex_str).decode('utf-8', errors='replace') # Replace invalid UTF-8 characters
    except (ValueError, TypeError):
        return None

# Colonne origin_decimal
df['origin_decimal'] = df['origin_account_stripped'].apply(hex_to_decimal)

# Colonne destination_decimal
df['destination_decimal'] = df['destination_account_stripped'].apply(hex_to_decimal)

# Colonne origin_utf8
df['origin_utf8'] = df['origin_account_stripped'].apply(hex_to_utf8)

# Colonne destination_utf8
df['destination_utf8'] = df['destination_account_stripped'].apply(hex_to_utf8)

print("New columns 'origin_decimal', 'destination_decimal', 'origin_utf8', 'destination_utf8' created.")
print(df[['origin_account', 'origin_account_stripped', 'origin_decimal', 'origin_utf8',
          'destination_account', 'destination_account_stripped', 'destination_decimal', 'destination_utf8']].head())

In [ ]:
import datetime

def extract_ipv4(hex_str):
    if len(hex_str) < 8: # l'IPv4 demande au moins 8 caractères hex
        return None
    try:
        # Les 8 derniers caractères pour l'IPv4
        hex_ip = hex_str[-8:]
        ip_parts = [str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2)]
        return ".".join(ip_parts)
    except (ValueError, TypeError):
        return None

def extract_mac(hex_str):
    if len(hex_str) < 12: # la MAC demande au moins 12 caractères hex
        return None
    try:
        # Les 12 derniers caractères pour l'adresse MAC (48 bits)
        hex_mac = hex_str[-12:]
        mac_parts = [hex_mac[i:i+2].upper() for i in range(0, 12, 2)]
        return ":".join(mac_parts)
    except (ValueError, TypeError):
        return None

def extract_timestamp_ns(hex_str):
    try:
        valeur_decimale = int(hex_str, 16)
        # Interprétation en horodatage Unix en nanosecondes
        timestamp_secondes = valeur_decimale / 1_000_000_000
        # Garde-fou contre les dates invalides, trop anciennes ou futures
        if 0 < timestamp_secondes < 4102444800: # du 1er janvier 1970 au 1er janvier 2100
            return datetime.datetime.fromtimestamp(timestamp_secondes, datetime.timezone.utc)
        else:
            return None
    except (ValueError, TypeError, OSError): # OSError sur un horodatage invalide
        return None

# Application aux comptes émetteurs
df['origin_ipv4'] = df['origin_account_stripped'].apply(extract_ipv4)
df['origin_mac'] = df['origin_account_stripped'].apply(extract_mac)
df['origin_timestamp'] = df['origin_account_stripped'].apply(extract_timestamp_ns)

# Application aux comptes destinataires
df['destination_ipv4'] = df['destination_account_stripped'].apply(extract_ipv4)
df['destination_mac'] = df['destination_account_stripped'].apply(extract_mac)
df['destination_timestamp'] = df['destination_account_stripped'].apply(extract_timestamp_ns)

In [ ]:
# Créer une colonne indiquant la présence d'un horodatage d'origine valide
df['has_origin_timestamp'] = df['origin_timestamp'].notna()

# Créer une colonne indiquant la présence d'un horodatage de destination valide
df['has_destination_timestamp'] = df['destination_timestamp'].notna()

print("Nouvelles colonnes 'has_origin_timestamp' et 'has_destination_timestamp' créées.")
print(df[['origin_timestamp', 'has_origin_timestamp', 'destination_timestamp', 'has_destination_timestamp', 'fraud_flag']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution du 'fraud_flag' par rapport à la présence d'un horodatage d'origine
plt.figure(figsize=(10, 6))
sns.countplot(x='has_origin_timestamp', hue='fraud_flag', data=df, palette='viridis')
plt.title('Distribution du Fraud Flag par Présence d\'horodatage d\'origine')
plt.xlabel('Présence d\'horodatage d\'origine (False = NaT, True = Valide)')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

# Distribution du 'fraud_flag' par rapport à la présence d'un horodatage de destination
plt.figure(figsize=(10, 6))
sns.countplot(x='has_destination_timestamp', hue='fraud_flag', data=df, palette='magma')
plt.title('Distribution du Fraud Flag par Présence d\'horodatage de destination')
plt.xlabel('Présence d\'horodatage de destination (False = NaT, True = Valide)')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

In [ ]:
all_ids_start_with_dtf = df['id'].apply(lambda x: x.startswith('dtf_')).all()

if all_ids_start_with_dtf:
    print("Oui, tous les IDs de transaction commencent par 'dtf_'.")
else:
    print("Non, tous les IDs de transaction ne commencent pas par 'dtf_'.")
    # Exemple de valeur non conforme
    non_conforming_id = df[~df['id'].apply(lambda x: x.startswith('dtf_'))]['id'].iloc[0]
    print(f"Exemple d'ID qui ne commence pas par 'dtf_': {non_conforming_id}")

In [ ]:
all_origin_accounts_start_with_acco = df['origin_account'].apply(lambda x: x.startswith('acc_o_')).all()

if all_origin_accounts_start_with_acco:
    print("Oui, tous les comptes émetteurs commencent par 'acc_o_'.")
else:
    print("Non, tous les comptes émetteurs ne commencent pas par 'acc_o_'.")
    # Exemple de valeur non conforme
    non_conforming_account = df[~df['origin_account'].apply(lambda x: x.startswith('acc_o_'))]['origin_account'].iloc[0]
    print(f"Exemple de compte qui ne commence pas par 'acc_o_': {non_conforming_account}")

In [ ]:
all_destination_accounts_start_with_accd = df['destination_account'].apply(lambda x: x.startswith('acc_d_')).all()

if all_destination_accounts_start_with_accd:
    print("Oui, tous les comptes récepteurs commencent par 'acc_d_'.")
else:
    print("Non, tous les comptes récepteurs ne commencent pas par 'acc_d_'.")
    # Exemple de valeur non conforme
    non_conforming_account = df[~df['destination_account'].apply(lambda x: x.startswith('acc_d_'))]['destination_account'].iloc[0]
    print(f"Exemple de compte qui ne commence pas par 'acc_d_': {non_conforming_account}")

In [ ]:
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

unique_stripped_origin = set(df['origin_account_stripped'].unique())
unique_stripped_destination = set(df['destination_account_stripped'].unique())

common_stripped_accounts = unique_stripped_origin.intersection(unique_stripped_destination)

if common_stripped_accounts:
    print(f"Oui, il y a {len(common_stripped_accounts)} comptes qui sont à la fois expéditeurs et récepteurs après avoir enlevé les préfixes.")
    print(f"Voici quelques exemples : {list(common_stripped_accounts)[:5]}")
else:
    print("Non, il n'y a pas de comptes qui sont à la fois expéditeurs et récepteurs après avoir enlevé les préfixes.")

### suite

In [ ]:
for col in df.columns:
  print(f'{col}:{df[col].nunique()}')

In [ ]:
#nombre de compte uniques expediteur ou receveur
a = df['origin_account'].to_list()
b = df['destination_account'].to_list()
c = a + b
print(f'nombre de compte uniques expediteur ou receveur: {len(np.unique(c))}')

In [ ]:
#le nombre d'expediteur etant aussi recepteur
# Trouve les éléments communs aux deux colonnes
valeurs_communes = df.loc[df['origin_account'].isin(df['destination_account']), 'origin_account'].unique()

print(f'le nombre d\'expediteur etant aussi recepteur: {len(valeurs_communes)}')

creer des colonnes historiques qui indique dans l'ordre l'apparution d'un destinataire puis une autre pour l'expediteur.

connaitre le nombre d'expediteur de flag 0 et 1, de meme pour destinataire

In [ ]:
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()

origin_accounts_in_both_fraud_and_non_fraud = set(non_fraudulent_origin_accounts).intersection(set(fraudulent_origin_accounts))
print(f"Number of unique origin accounts involved in both fraud (flag 1) and non-fraud (flag 0) transactions: {len(origin_accounts_in_both_fraud_and_non_fraud)}")
print(f'nombre de compte unique frauduleux: {len(fraudulent_origin_accounts)}')
print(f'nombre de compte unique non frauduleux: {len(non_fraudulent_origin_accounts)}')

In [ ]:
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

destination_accounts_in_both_fraud_and_non_fraud = set(non_fraudulent_destination_accounts).intersection(set(fraudulent_destination_accounts))
print(f"Number of unique destination accounts involved in both fraud (flag 1) and non-fraud (flag 0) transactions: {len(destination_accounts_in_both_fraud_and_non_fraud)}")
print(f'nombre de compte unique frauduleux (destination): {len(fraudulent_destination_accounts)}')
print(f'nombre de compte unique non frauduleux (destination): {len(non_fraudulent_destination_accounts)}')


In [ ]:
# Transactions frauduleuses (fraud_flag == 1)
fraudulent_transactions = df[df['fraud_flag'] == 1]

# Nombre de fraudes par origin_account
fraud_counts_per_origin_account = fraudulent_transactions['origin_account'].value_counts()

# Minimum et maximum de fraudes par compte émetteur frauduleux
min_frauds = fraud_counts_per_origin_account.min()
max_frauds = fraud_counts_per_origin_account.max()

print(f"Minimum number of frauds for a unique fraudulent origin account: {min_frauds}")
print(f"Maximum number of frauds for a unique fraudulent origin account: {max_frauds}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(fraud_counts_per_origin_account, bins=40, kde=True)
plt.title('Distribution du nombre de fraudes par compte émetteur frauduleux')
plt.xlabel('Nombre de fraudes par compte émetteur')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
# Transactions frauduleuses (fraud_flag == 1)

# Nombre de fraudes par destination_account
fraud_counts_per_destination_account = fraudulent_transactions['destination_account'].value_counts()

# Minimum et maximum de fraudes par compte destinataire frauduleux
min_frauds_dest = fraud_counts_per_destination_account.min()
max_frauds_dest = fraud_counts_per_destination_account.max()

print(f"Minimum number of frauds for a unique fraudulent destination account: {min_frauds_dest}")
print(f"Maximum number of frauds for a unique fraudulent destination account: {max_frauds_dest}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(fraud_counts_per_destination_account, bins=30, kde=True)
plt.title('Distribution du nombre de fraudes par compte destinataire frauduleux')
plt.xlabel('Nombre de fraudes par compte destinataire')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
df = df.sort_values(by='period').reset_index(drop=True)
print("DataFrame 'df' has been sorted by the 'period' column.")
df[['id', 'period', 'operation']].head()

In [ ]:
print(f"Total number of fraudulent transactions (fraud_flag = 1): {df['fraud_flag'].sum()}")
print(f"Number of unique transaction IDs with fraud_flag = 1: {df[df['fraud_flag'] == 1]['id'].nunique()}")

# Comptes émetteurs impliqués dans une fraude
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
print(f"\nNumber of unique origin accounts involved in fraud: {len(fraudulent_origin_accounts)}")

# Comptes destinataires impliqués dans une fraude
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
print(f"Number of unique destination accounts involved in fraud: {len(fraudulent_destination_accounts)}")

# Les comptes émetteurs frauduleux apparaissent-ils aussi hors fraude ?
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
origin_fraud_and_non_fraud = set(fraudulent_origin_accounts).intersection(set(non_fraudulent_origin_accounts))
print(f"\nNumber of origin accounts involved in both fraudulent and non-fraudulent transactions: {len(origin_fraud_and_non_fraud)}")

# Les comptes destinataires frauduleux apparaissent-ils aussi hors fraude ?
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
destination_fraud_and_non_fraud = set(fraudulent_destination_accounts).intersection(set(non_fraudulent_destination_accounts))
print(f"Number of destination accounts involved in both fraudulent and non-fraudulent transactions: {len(destination_fraud_and_non_fraud)}")

# Ensemble des comptes ayant fraudé au moins une fois (émetteur ou destinataire)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Drapeaux d'historique de fraude pour l'émetteur et le destinataire
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("\nNew columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

In [ ]:
df['operation'].value_counts()

probleme de la fraude est la destination

emetteur honnete qui envoie vers distination fraudeuleuse est une fraude

les destinataire frauduleux ne font jamais les autres operations

-fraude 03: transfert sans arrivé sur le compte ou departs

les seules fraudes sont celles de compte recepteurs et jamais de compte emetteur

01: retrait, négatif autorisé

02: aucune variation de montant expediteur ou recepteurs

03: transfer ou prets, transfert négatif restant autorisé, les comptes destinataires ne font jamais une autre transactions fraude ou non

04: depot

05: retrait par code

op_04, op_02, op_05: toujours ensembles sur un compte destinataire non-frauduleux


In [ ]:
first_fraud_occurrence = df[df['fraud_flag'] == 1]['period'].min()
print(f"La première apparition d'une transaction frauduleuse est à la période: {first_fraud_occurrence}")

In [ ]:
first_fraud_row_index = df[df['fraud_flag'] == 1].index[0]
print(f"Le numéro de la ligne de la première apparition d'une transaction frauduleuse est: {first_fraud_row_index}")

### Analyse des Comptes Destinataires : Transition vers la Fraude

In [ ]:
# 1. Identifier les comptes destinataires qui sont non-frauduleux puis frauduleux

# Minimum et maximum de fraud_flag par compte destinataire,
# et période de la première transaction non frauduleuse puis frauduleuse
account_fraud_summary = df.groupby('destination_account').agg(
    first_fraud_period=('period', lambda x: x[df.loc[x.index, 'fraud_flag'] == 1].min()),
    first_non_fraud_period=('period', lambda x: x[df.loc[x.index, 'fraud_flag'] == 0].min()),
    has_fraud=('fraud_flag', lambda x: (x == 1).any()),
    has_non_fraud=('fraud_flag', lambda x: (x == 0).any())
).reset_index()

# Comptes non frauduleux puis devenus frauduleux
also_become_fraudulent = account_fraud_summary[
    account_fraud_summary['has_non_fraud'] &
    account_fraud_summary['has_fraud'] &
    (account_fraud_summary['first_non_fraud_period'] < account_fraud_summary['first_fraud_period'])
]

print(f"Nombre de comptes destinataires qui étaient non-frauduleux (flag=0) au début et sont devenus frauduleux (flag=1) plus tard : {len(also_become_fraudulent)}")
if not also_become_fraudulent.empty:
    print("Exemples de tels comptes :\n", also_become_fraudulent.head())
else:
    print("Aucun compte destinataire n'a été trouvé qui est passé de non-frauduleux à frauduleux.")

### Analyse des Comptes Destinataires : Persistance de la Fraude

In [ ]:
# 2. Vérifier si un compte, une fois devenu frauduleux, le reste toujours

# Comptes ayant au moins une transaction frauduleuse en destinataire
fraudulent_dest_accounts_overall = df[df['fraud_flag'] == 1]['destination_account'].unique()

always_fraudulent_once_fraud = []

for account in fraudulent_dest_accounts_overall:
    account_transactions = df[df['destination_account'] == account].sort_values(by='period')

    # Période de la première transaction frauduleuse
    first_fraud_period_for_account = account_transactions[account_transactions['fraud_flag'] == 1]['period'].min()

    # Transactions postérieures à la première fraude
    subsequent_transactions = account_transactions[account_transactions['period'] > first_fraud_period_for_account]

    # S'il existe des transactions ultérieures non frauduleuses, le compte n'est pas toujours frauduleux
    if not subsequent_transactions.empty and (subsequent_transactions['fraud_flag'] == 0).any():
        always_fraudulent_once_fraud.append(False)
    else:
        # Sinon, le compte reste toujours frauduleux
        always_fraudulent_once_fraud.append(True)

# Part des comptes restant toujours frauduleux
if fraudulent_dest_accounts_overall.size > 0:
    num_always_fraud = sum(always_fraudulent_once_fraud)
    print(f"Sur {len(fraudulent_dest_accounts_overall)} comptes destinataires ayant commis la fraude au moins une fois :")
    print(f"- {num_always_fraud} comptes sont restés frauduleux après leur première fraude.")
    print(f"- {len(fraudulent_dest_accounts_overall) - num_always_fraud} comptes ont eu des transactions non-frauduleuses après leur première fraude.")
else:
    print("Aucun compte destinataire n'a été impliqué dans la fraude.")

Sur 5000 comptes destinataires ayant commis la fraude au moins une fois :
- 0 comptes sont restés frauduleux après leur première fraude.
- 5000 comptes ont eu des transactions non-frauduleuses après leur première fraude.

Nombre de comptes destinataires qui étaient non-frauduleux (flag=0) au début et sont devenus frauduleux (flag=1) plus tard : 2549

il y a des comptes recpeteurs au debut qui ne sont pas des fraudes et qui deviendront fraude.

est ce que dé devenu fraude il va rester ainsi?

les comptes fraudes recepteurs apparaise la premier fois avec 85 a 100 et subitement de grande somme

gros transfert d'un expediteur qui n a rien et parfois des chiffres negatif vers un compte vide. CELA AU DEBUT

dans 03 peut on faire du retrait negatif dans non fraude?

### K-Means sur les données `op_03`

Recherche d'une séparation dans les transactions `op_03` par clustering K-Means sur trois features numériques : `amount`, `origin_balance_after` et `destination_balance_after`.

Extraction des transactions `op_03`, standardisation pour que chaque feature pèse autant dans le calcul de distance, clustering, puis visualisation 3D.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Restriction à operation == op_03, avec copie
df_op03_clustering = df[df['operation'] == 'op_03'].copy()

# Features numériques pour le clustering
features_for_clustering = ['amount', 'origin_balance_after', 'destination_balance_after']

# Suppression des NaN avant KMeans
df_op03_clustering.dropna(subset=features_for_clustering, inplace=True)

# Vérification que le DataFrame n'est pas vide
if df_op03_clustering.empty:
    print("No data available for clustering after filtering 'op_03' and dropping NaNs.")
else:
    X_clustering = df_op03_clustering[features_for_clustering]

    # Standardisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clustering)

    # K-Means à 2 clusters, comme fraud_flag est binaire
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10) # n_init fixé explicitement
    df_op03_clustering['cluster_label'] = kmeans.fit_predict(X_scaled)

    print("K-Means clustering completed. Visualizing results...")

    # Visualisation 3D des clusters
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(
        df_op03_clustering['amount'],
        df_op03_clustering['origin_balance_after'],
        df_op03_clustering['destination_balance_after'],
        c=df_op03_clustering['cluster_label'],
        cmap='viridis', # Different color for each cluster
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D K-Means Clusters for op_03 Transactions')

    # Barre de couleur
    legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
    ax.add_artist(legend1)

    plt.show()

    # Visualisation 3D de fraud_flag, pour comparaison
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    scatter_fraud = ax.scatter(
        df_op03_clustering['amount'],
        df_op03_clustering['origin_balance_after'],
        df_op03_clustering['destination_balance_after'],
        c=df_op03_clustering['fraud_flag'],
        cmap='coolwarm', # Different color for fraud vs non-fraud
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D Fraud Flag Distribution for op_03 Transactions')

    # Barre de couleur
    legend2 = ax.legend(*scatter_fraud.legend_elements(), title="Fraud Flag")
    ax.add_artist(legend2)

    plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

if not df_op03_clustering.empty:
    # Score de silhouette
    silhouette_avg = silhouette_score(X_scaled, df_op03_clustering['cluster_label'])
    print(f"The average silhouette score for the K-Means clusters is: {silhouette_avg:.4f}")
else:
    print("Cannot calculate silhouette score: No data available for clustering.")

Visualisation 3D des transactions `op_03` selon `amount`, `origin_balance_after` et `destination_balance_after`. Le premier graphe colore les points par cluster K-Means, le second par `fraud_flag` réel.

La comparaison des deux dit si les clusters recoupent la frontière fraude / non-fraude. Une séparation nette ferait de ces trois features de bons indicateurs.

### Distribution des clusters K-Means par `fraud_flag`

Mesure de la correspondance entre les clusters et les transactions frauduleuses.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='cluster_label', hue='fraud_flag', data=df_op03_clustering, palette='coolwarm')
plt.title('Distribution des clusters K-Means par Fraud Flag pour op_03')
plt.xlabel('Cluster K-Means')
plt.ylabel('Nombre de Transactions')
plt.legend(title='Fraud Flag')
plt.show()

### DBSCAN sur le cluster 1 du K-Means

Second passage de clustering, en DBSCAN cette fois, sur les transactions classées dans le `cluster_label` 1. DBSCAN trouve des clusters de forme arbitraire et isole le bruit : il peut révéler des structures que K-Means écrase.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# --- K-Means, pour définir df_op03_clustering ---
# Restriction à operation == op_03, avec copie
df_op03_clustering = df[df['operation'] == 'op_03'].copy()

# Features numériques pour le clustering
features_for_clustering = ['amount', 'origin_balance_after', 'destination_balance_after']

# Suppression des NaN avant KMeans
df_op03_clustering.dropna(subset=features_for_clustering, inplace=True)

# Vérification que le DataFrame n'est pas vide
if df_op03_clustering.empty:
    print("No data available for K-Means clustering after filtering 'op_03' and dropping NaNs.")
    # Sortie si aucune donnée
    exit()
else:
    X_clustering = df_op03_clustering[features_for_clustering]

    # Standardisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clustering)

    # K-Means à 2 clusters, comme fraud_flag est binaire
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    df_op03_clustering['cluster_label'] = kmeans.fit_predict(X_scaled)

    print("K-Means clustering completed as a prerequisite for DBSCAN.")

# --- DBSCAN ---
# Cluster 1 du K-Means précédent
df_cluster1 = df_op03_clustering[df_op03_clustering['cluster_label'] == 1].copy()

# Assez de données dans le cluster 1 pour un second clustering ?
if df_cluster1.empty:
    print("No data in cluster_label 1 to perform further clustering with DBSCAN.")
else:
    # Mêmes features de clustering
    features_for_clustering_dbscan = ['amount', 'origin_balance_after', 'destination_balance_after']
    X_cluster1 = df_cluster1[features_for_clustering_dbscan]

    # Standardisation, indispensable pour DBSCAN
    scaler_dbscan = StandardScaler()
    X_scaled_cluster1 = scaler_dbscan.fit_transform(X_cluster1)

    # DBSCAN
    # eps et min_samples à régler selon la densité des données
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    df_cluster1['dbscan_cluster_label'] = dbscan.fit_predict(X_scaled_cluster1)

    print(f"DBSCAN clustering applied to K-Means Cluster 1. Found {df_cluster1['dbscan_cluster_label'].nunique()} clusters (including noise -1).")

    # Visualisation 3D des clusters DBSCAN dans le cluster 1 du K-Means
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Palette distincte pour les clusters DBSCAN
    scatter = ax.scatter(
        df_cluster1['amount'],
        df_cluster1['origin_balance_after'],
        df_cluster1['destination_balance_after'],
        c=df_cluster1['dbscan_cluster_label'],
        cmap='plasma', # Another colormap for differentiation
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D DBSCAN Clusters within K-Means Cluster 1 (op_03 Transactions)')

    # Barre de couleur
    legend1 = ax.legend(*scatter.legend_elements(), title="DBSCAN Clusters")
    ax.add_artist(legend1)

    plt.show()

    # Distribution de fraud_flag par cluster DBSCAN dans le cluster 1 du K-Means
    plt.figure(figsize=(10, 6))
    sns.countplot(x='dbscan_cluster_label', hue='fraud_flag', data=df_cluster1, palette='viridis')
    plt.title('Distribution des clusters DBSCAN par Fraud Flag (dans K-Means Cluster 1)')
    plt.xlabel('Cluster DBSCAN')
    plt.ylabel('Nombre de Transactions')
    plt.legend(title='Fraud Flag')
    plt.show()

### Corrélation entre clusters DBSCAN et `fraud_flag`

Corrélation de Pearson entre `dbscan_cluster_label` et `fraud_flag` : plus elle est forte, en positif ou en négatif, plus les clusters captent la fraude.

In [ ]:
if 'df_cluster1' in locals() and not df_cluster1.empty:
    correlation_dbscan_fraud = df_cluster1[['dbscan_cluster_label', 'fraud_flag']].corr()
    print("Matrice de corrélation entre les clusters DBSCAN et le fraud_flag :")
    display(correlation_dbscan_fraud)
else:
    print("Le DataFrame df_cluster1 n'est pas défini ou est vide, impossible de calculer la corrélation.")

In [ ]:
df03 = df[df['operation'] == 'op_03'].copy()
df03['delta_origin_account'] = df03['origin_balance_after'] - df03['origin_balance_before']
df03['delta_destination_account'] = df03['destination_balance_after'] - df03['destination_balance_before']
print("DataFrame df03 created with 'op_03' transactions and new delta columns.")
df03[['operation', 'origin_balance_before', 'origin_balance_after', 'delta_origin_account', 'destination_balance_before', 'destination_balance_after', 'delta_destination_account']].head()

la diffrence des deltas est toujours simultanément 0 ou toulours simultaément different

In [ ]:
# Distribution de fraud_flag quand delta_origin_account vaut 0
print("\nDistribution de fraud_flag lorsque delta_origin_account est different de 0:")
df03_delta_origin_zero = df03[df03['delta_origin_account'] < 0]
print(df03_delta_origin_zero['fraud_flag'].value_counts(normalize= True))
print(df03_delta_origin_zero['fraud_flag'].value_counts())

# Distribution de fraud_flag quand delta_destination_account vaut 0
print("\nDistribution de fraud_flag lorsque delta_destination_account est 0:")
df03_delta_dest_zero = df03[df03['delta_destination_account'] == 0]
print(df03_delta_dest_zero['fraud_flag'].value_counts(normalize= True))
print(df03_delta_dest_zero['fraud_flag'].value_counts())


In [ ]:
# Comptes destinataires distincts avec delta_destination_account nul
dest_accounts_delta_zero = set(df03[df03['delta_destination_account'] == 0]['destination_account'].unique())

# Comptes destinataires distincts avec delta_destination_account non nul
dest_accounts_delta_not_zero = set(df03[df03['delta_destination_account'] != 0]['destination_account'].unique())

# Intersection des deux ensembles
intersecting_accounts = dest_accounts_delta_zero.intersection(dest_accounts_delta_not_zero)

if intersecting_accounts:
    print(f"Oui, il y a {len(intersecting_accounts)} comptes récepteurs uniques qui ont un delta = 0 et un delta != 0.")
    print("Voici quelques exemples de ces comptes :", list(intersecting_accounts)[:5])
else:
    print("Non, il n'y a pas de comptes récepteurs uniques qui ont un delta = 0 et un delta != 0.")

In [ ]:
# 1. Comptes à delta variable (intersecting_accounts)
variable_delta_dest_accounts = intersecting_accounts

# 2. Comptes à delta nul : uniquement delta_destination_account == 0 en op_03
all_dest_accounts_op03 = set(df03['destination_account'].unique())
fixed_zero_delta_dest_accounts = set(dest_accounts_delta_zero) - variable_delta_dest_accounts

print(f"\nNumber of destination accounts with variable delta: {len(variable_delta_dest_accounts)}")
print(f"Number of destination accounts with fixed (zero) delta: {len(fixed_zero_delta_dest_accounts)}")

In [ ]:
# 3. Fraude des comptes à delta variable
print("\n--- Analyse des comptes avec delta variable ---")
df_variable_delta = df03[df03['destination_account'].isin(list(variable_delta_dest_accounts))]
if not df_variable_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta variable:")
    print(df_variable_delta['fraud_flag'].value_counts(normalize=True))
else:
    print("Aucune transaction trouvée pour les comptes à delta variable.")

In [ ]:
# 4. Fraude des comptes à delta nul
print("\n--- Analyse des comptes avec delta fixe (zéro) ---")
df_fixed_zero_delta = df03[df03['destination_account'].isin(list(fixed_zero_delta_dest_accounts))]
if not df_fixed_zero_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta fixe (zéro):")
    print(df_fixed_zero_delta['fraud_flag'].value_counts())
else:
    print("Aucune transaction trouvée pour les comptes à delta fixe (zéro).")

In [ ]:
# Comptes n'ayant que des deltas non nuls
# Présents dans dest_accounts_delta_not_zero mais absents de dest_accounts_delta_zero
fixed_non_zero_delta_dest_accounts = set(dest_accounts_delta_not_zero) - set(dest_accounts_delta_zero)

print(f"\nNumber of destination accounts with fixed (non-zero) delta: {len(fixed_non_zero_delta_dest_accounts)}")

# Fraude des comptes à delta fixe non nul
print("\n--- Analyse des comptes avec delta fixe (non-zéro) ---")
df_fixed_non_zero_delta = df03[df03['destination_account'].isin(list(fixed_non_zero_delta_dest_accounts))]

if not df_fixed_non_zero_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta fixe (non-zéro):")
    print(df_fixed_non_zero_delta['fraud_flag'].value_counts(normalize=True))
else:
    print("Aucune transaction trouvée pour les comptes à delta fixe (non-zéro).")

In [ ]:
zero_origin_delta = df03[df03['delta_origin_account'] == 0]

if not zero_origin_delta.empty:
    unique_dest_deltas = zero_origin_delta['delta_destination_account'].unique()
    if len(unique_dest_deltas) == 1 and unique_dest_deltas[0] == 0:
        print("Oui, à chaque fois que le delta de l'émetteur est 0, celui du récepteur est aussi 0 pour les transactions 'op_03'.")
    else:
        print("Non, il y a des cas où le delta de l'émetteur est 0, mais celui du récepteur n'est pas 0 (ou vice versa) pour les transactions 'op_03'.")
        print("Valeurs uniques de delta_destination_account lorsque delta_origin_account est 0:")
        print(unique_dest_deltas)
else:
    print("Aucune transaction trouvée où le delta de l'émetteur est 0 dans 'op_03'.")

In [ ]:
df.head(20)

In [ ]:
fraud_with_non_fraud_dest = df[(df['fraud_flag'] == 1) & (df['destination_account_previously_fraud'] == 0)]

num_fraud_with_non_fraud_dest = len(fraud_with_non_fraud_dest)

print(f"Number of fraudulent transactions where destination_account_previously_fraud is 0: {num_fraud_with_non_fraud_dest}")

if num_fraud_with_non_fraud_dest > 0:
    print("Yes, there are fraudulent transactions where the destination account was not previously flagged as fraudulent.")
else:
    print("No, there are no fraudulent transactions where the destination account was not previously flagged as fraudulent.")

# Quelques exemples
if num_fraud_with_non_fraud_dest > 0:
    print("\nExamples of such fraudulent transactions:")
    print(fraud_with_non_fraud_dest.head())

In [ ]:
fraudulent_destination_per_operation = df[df['fraud_flag'] == 0].groupby('operation')['destination_account'].nunique()
print("Répartition des comptes destinataires non-frauduleux uniques par opération :")
print(fraudulent_destination_per_operation)

In [ ]:
# 1. Comptes jamais frauduleux

# 2. Transactions vers ces comptes destinataires jamais frauduleux
df_truly_non_fraud_dest_txns = df[df['destination_account'].isin(truly_non_fraudulent_accounts)]

# 3. Types d'opérations distincts par compte destinataire
operations_set_per_truly_non_fraud_dest_account = df_truly_non_fraud_dest_txns.groupby('destination_account')['operation'].apply(lambda x: set(x.unique()))

# 4. Comptes impliqués dans op_03
accounts_involved_in_op03 = operations_set_per_truly_non_fraud_dest_account[operations_set_per_truly_non_fraud_dest_account.apply(lambda x: 'op_03' in x)]

# 5. Parmi eux, combien font op_03 et au moins une autre opération
#    (ensemble d'opérations de taille supérieure à 1)
accounts_in_op03_and_multiple_others = accounts_involved_in_op03[accounts_involved_in_op03.apply(len) > 1]

print(f"Number of unique non-fraudulent destination accounts involved in 'op_03' AND at least one other operation: {len(accounts_in_op03_and_multiple_others)}")
print("These accounts are:")
print(accounts_in_op03_and_multiple_others)

In [ ]:
fraudulent_destination_per_operation = df[df['fraud_flag'] == 1].groupby('operation')['destination_account'].nunique()
print("Répartition des comptes destinataires frauduleux uniques par opération :")
print(fraudulent_destination_per_operation)

In [ ]:
# 1. Comptes émetteurs frauduleux et leur nombre de transactions

# Transactions issues de comptes émetteurs frauduleux
df_fraud_origin_txns = df[df['origin_account'].isin(fraudulent_origin_accounts)]
# Nombre de transactions par compte émetteur frauduleux
fraud_origin_tx_counts = df_fraud_origin_txns['origin_account'].value_counts()

# 2. Comptes émetteurs jamais frauduleux et leur nombre de transactions

# Transactions issues de comptes émetteurs jamais frauduleux
df_non_fraud_origin_txns = df[df['origin_account'].isin(truly_non_fraudulent_accounts)]
# Nombre de transactions par compte émetteur non frauduleux
non_fraud_origin_tx_counts = df_non_fraud_origin_txns['origin_account'].value_counts()

# 3. Comptes les plus actifs de chaque groupe

# Top 3 des comptes émetteurs frauduleux par nombre de transactions
top3_fraud_origin_accounts = fraud_origin_tx_counts.head(3).index.tolist()

# fraud1e, fraud2e, fraud3e
if len(top3_fraud_origin_accounts) > 0:
    fraud1e = df[df['origin_account'] == top3_fraud_origin_accounts[0]]
    print(f"Created fraud1e for account {top3_fraud_origin_accounts[0]} with {len(fraud1e)} entries.")
else:
    print("No fraudulent origin accounts found to create fraud1e.")

if len(top3_fraud_origin_accounts) > 1:
    fraud2e = df[df['origin_account'] == top3_fraud_origin_accounts[1]]
    print(f"Created fraud2e for account {top3_fraud_origin_accounts[1]} with {len(fraud2e)} entries.")

if len(top3_fraud_origin_accounts) > 2:
    fraud3e = df[df['origin_account'] == top3_fraud_origin_accounts[2]]
    print(f"Created fraud3e for account {top3_fraud_origin_accounts[2]} with {len(fraud3e)} entries.")

# Top 3 des comptes émetteurs jamais frauduleux par nombre de transactions
top3_non_fraud_origin_accounts = non_fraud_origin_tx_counts.head(3).index.tolist()

# nonfraud1e, nonfraud2e, nonfraud3e
if len(top3_non_fraud_origin_accounts) > 0:
    nonfraud1e = df[df['origin_account'] == top3_non_fraud_origin_accounts[0]]
    print(f"Created nonfraud1e for account {top3_non_fraud_origin_accounts[0]} with {len(nonfraud1e)} entries.")
else:
    print("No truly non-fraudulent origin accounts found to create nonfraud1e.")

if len(top3_non_fraud_origin_accounts) > 1:
    nonfraud2e = df[df['origin_account'] == top3_non_fraud_origin_accounts[1]]
    print(f"Created nonfraud2e for account {top3_non_fraud_origin_accounts[1]} with {len(nonfraud2e)} entries.")

if len(top3_non_fraud_origin_accounts) > 2:
    nonfraud3e = df[df['origin_account'] == top3_non_fraud_origin_accounts[2]]
    print(f"Created nonfraud3e for account {top3_non_fraud_origin_accounts[2]} with {len(nonfraud3e)} entries.")

In [ ]:
nonfraud3e.head(20)

In [ ]:
fraud1e.head(20)

si le amount superieur a origin_balance_before alors fraude

operation de type 04 depot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fraudulent_origin_dfs = {'fraud1e': fraud1e, 'fraud2e': fraud2e, 'fraud3e': fraud3e}

for name, df_account in fraudulent_origin_dfs.items():
    if not df_account.empty:
        # ID du compte émetteur pour le titre
        account_id = df_account['origin_account'].iloc[0]

        # Périodes frauduleuses marquées par des lignes verticales
        fraud_periods = df_account[df_account['fraud_flag'] == 1]['period'].unique()

        # Tracé pour origin_balance_after
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_after', data=df_account, label='Solde du compte émetteur (après transaction)', marker='o', markersize=4, color='blue')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Solde après transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (après transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Tracé pour origin_balance_before
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_before', data=df_account, label='Solde du compte émetteur (avant transaction)', marker='x', markersize=4, color='orange')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Solde avant transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (avant transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Tracé pour amount
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='amount', data=df_account, label='Montant de la transaction', marker='s', markersize=4, color='purple')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Montant de la transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Montant de la transaction')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        print(f"DataFrame {name} est vide, impossible de visualiser.")

### Détail des comptes non frauduleux

Soldes avant, soldes après et montants séparés pour `nonfraud1e`, `nonfraud2e` et `nonfraud3e`. Ces comptes n'ayant jamais fraudé, aucun marqueur de fraude n'apparaît.

In [ ]:
non_fraudulent_origin_dfs = {'nonfraud1e': nonfraud1e, 'nonfraud2e': nonfraud2e, 'nonfraud3e': nonfraud3e}

for name, df_account in non_fraudulent_origin_dfs.items():
    if not df_account.empty:
        account_id = df_account['origin_account'].iloc[0]

        # Tracé pour origin_balance_after
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_after', data=df_account, label='Solde du compte émetteur (après transaction)', marker='o', markersize=4, color='blue')
        plt.title(f'Solde après transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (après transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Tracé pour origin_balance_before
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_before', data=df_account, label='Solde du compte émetteur (avant transaction)', marker='x', markersize=4, color='orange')
        plt.title(f'Solde avant transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (avant transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Tracé pour amount
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='amount', data=df_account, label='Montant de la transaction', marker='s', markersize=4, color='purple')
        plt.title(f'Montant de la transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Montant de la transaction')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        print(f"DataFrame {name} est vide, impossible de visualiser.")

## test rapide

In [ ]:
df.describe(include='all')

## Distributions par `fraud_flag`

Distributions des variables clés ventilées par `fraud_flag`, à la recherche de motifs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore') # Suppress warnings for better readability

# Boîte à moustaches de period
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='period', data=df)
plt.title('Period Distribution by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Period')
plt.show()

In [ ]:
# Diagramme en barres de operation
plt.figure(figsize=(10, 6))
sns.countplot(x='operation', hue='fraud_flag', data=df, palette='viridis')
plt.title('Operation Type Distribution by Fraud Flag')
plt.xlabel('Operation Type')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

In [ ]:
# Boîte à moustaches de amount
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='amount', data=df)
plt.title('Transaction Amount Distribution by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Amount')
plt.ylim(0, df['amount'].quantile(0.99)) # limite l'axe y sur la distribution principale
plt.show()

In [ ]:
# Boîte à moustaches de origin_balance_before
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_before', data=df)
plt.title('Origin Balance Before Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance Before')
plt.ylim(df['origin_balance_before'].quantile(0.01), df['origin_balance_before'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de origin_balance_after
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_after', data=df)
plt.title('Origin Balance After Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance After')
plt.ylim(df['origin_balance_after'].quantile(0.01), df['origin_balance_after'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de destination_balance_before
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_before', data=df)
plt.title('Destination Balance Before Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance Before')
plt.ylim(df['destination_balance_before'].quantile(0.01), df['destination_balance_before'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de destination_balance_after
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_after', data=df)
plt.title('Destination Balance After Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance After')
plt.ylim(df['destination_balance_after'].quantile(0.01), df['destination_balance_after'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

## Distributions sur `operation = 'op_03'`, par `fraud_flag`

`op_03` étant le type d'opération critique pour la fraude, on reprend les distributions des autres variables sur ce seul périmètre.

In [ ]:
# Restriction à operation == op_03
df_op03 = df[df['operation'] == 'op_03'].copy()

print(f"Filtered DataFrame shape (only op_03): {df_op03.shape}")

In [ ]:
# Boîte à moustaches de period, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='period', data=df_op03)
plt.title('Period Distribution for op_03 Transactions by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Period')
plt.show()

In [ ]:
# Boîte à moustaches de amount, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='amount', data=df_op03)
plt.title('Transaction Amount Distribution for op_03 Transactions by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Amount')
plt.ylim(0, df_op03['amount'].quantile(0.99)) # limite l'axe y sur la distribution principale
plt.show()

In [ ]:
# Boîte à moustaches de origin_balance_before, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_before', data=df_op03)
plt.title('Origin Balance Before Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance Before')
plt.ylim(df_op03['origin_balance_before'].quantile(0.01), df_op03['origin_balance_before'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de origin_balance_after, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_after', data=df_op03)
plt.title('Origin Balance After Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance After')
plt.ylim(df_op03['origin_balance_after'].quantile(0.01), df_op03['origin_balance_after'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de destination_balance_before, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_before', data=df_op03)
plt.title('Destination Balance Before Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance Before')
plt.ylim(df_op03['destination_balance_before'].quantile(0.01), df_op03['destination_balance_before'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
# Boîte à moustaches de destination_balance_after, restreint à op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_after', data=df_op03)
plt.title('Destination Balance After Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance After')
plt.ylim(df_op03['destination_balance_after'].quantile(0.01), df_op03['destination_balance_after'].quantile(0.99)) # limite l'axe y pour la lisibilité
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution de fraud_flag selon origin_account_previously_fraud
plt.figure(figsize=(10, 6))
sns.countplot(x='origin_account_previously_fraud', hue='fraud_flag', data=df, palette='pastel')
plt.title('Fraud Flag Distribution by Origin Account Previously Fraud Status')
plt.xlabel('Origin Account Previously Fraud (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Distribution de fraud_flag selon destination_account_previously_fraud
plt.figure(figsize=(10, 6))
sns.countplot(x='destination_account_previously_fraud', hue='fraud_flag', data=df, palette='pastel')
plt.title('Fraud Flag Distribution by Destination Account Previously Fraud Status')
plt.xlabel('Destination Account Previously Fraud (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Fusion des deux colonnes en une variable catégorielle
df['fraud_history_status'] = df['origin_account_previously_fraud'].astype(str) + df['destination_account_previously_fraud'].astype(str)

# Distribution de fraud_flag selon l'historique de fraude combiné
plt.figure(figsize=(12, 7))
sns.countplot(x='fraud_history_status', hue='fraud_flag', data=df, palette='viridis')
plt.title('Fraud Flag Distribution by Combined Fraud History Status (Origin-Destination)')
plt.xlabel('Fraud History Status (Origin_Fraud_Prev | Destination_Fraud_Prev)')
plt.ylabel('Count')
plt.show()

print("Distribution analysis for new features complete. The 'fraud_history_status' column has been created for combined analysis.")

## EDA

In [ ]:
print(f"Total number of fraudulent transactions (fraud_flag = 1): {df['fraud_flag'].sum()}")
print(f"Number of unique transaction IDs with fraud_flag = 1: {df[df['fraud_flag'] == 1]['id'].nunique()}")

# Comptes émetteurs impliqués dans une fraude
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
print(f"\nNumber of unique origin accounts involved in fraud: {len(fraudulent_origin_accounts)}")

# Comptes destinataires impliqués dans une fraude
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
print(f"Number of unique destination accounts involved in fraud: {len(fraudulent_destination_accounts)}")

# Les comptes émetteurs frauduleux apparaissent-ils aussi hors fraude ?
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
origin_fraud_and_non_fraud = set(fraudulent_origin_accounts).intersection(set(non_fraudulent_origin_accounts))
print(f"\nNumber of origin accounts involved in both fraudulent and non-fraudulent transactions: {len(origin_fraud_and_non_fraud)}")

# Les comptes destinataires frauduleux apparaissent-ils aussi hors fraude ?
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
destination_fraud_and_non_fraud = set(fraudulent_destination_accounts).intersection(set(non_fraudulent_destination_accounts))
print(f"Number of destination accounts involved in both fraudulent and non-fraudulent transactions: {len(destination_fraud_and_non_fraud)}")

# Ensemble des comptes ayant fraudé au moins une fois (émetteur ou destinataire)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Drapeaux d'historique de fraude pour l'émetteur et le destinataire
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("\nNew columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

les comptes frauduleux sont dans le dataset dans le groupe frauduleux et non frauduleux

In [ ]:
are_all_fraudulent_destination_also_origin = set(fraudulent_destination_accounts).issubset(set(fraudulent_origin_accounts))
print(f"Are all fraudulent destination accounts also fraudulent origin accounts? {are_all_fraudulent_destination_also_origin}")

In [ ]:
fraudulent_origin_and_destination_accounts = set(fraudulent_origin_accounts).intersection(set(fraudulent_destination_accounts))
print(f"Number of accounts that are both fraudulent origin and fraudulent destination accounts: {len(fraudulent_origin_and_destination_accounts)}")

# Ces comptes jouent-ils les deux rôles, émetteur et destinataire ?

# Transactions dont l'émetteur ou le destinataire est dans fraudulent_origin_and_destination_accounts
df_dual_role_accounts_transactions = df[
    df['origin_account'].isin(fraudulent_origin_and_destination_accounts) |
    df['destination_account'].isin(fraudulent_origin_and_destination_accounts)
].copy()

print(f"\nTotal transactions involving accounts that are both fraudulent origin and destination: {len(df_dual_role_accounts_transactions)}")

# Ces comptes à double rôle envoient-ils et reçoivent-ils de l'argent ?
sends_money = df_dual_role_accounts_transactions['origin_account'].isin(fraudulent_origin_and_destination_accounts).any()
receives_money = df_dual_role_accounts_transactions['destination_account'].isin(fraudulent_origin_and_destination_accounts).any()
print(f"Do these dual-role accounts send money in these transactions? {sends_money}")
print(f"Do these dual-role accounts receive money in these transactions? {receives_money}")

# Part des transactions frauduleuses parmi celles des comptes à double rôle
if not df_dual_role_accounts_transactions.empty:
    fraud_proportion_dual_role = df_dual_role_accounts_transactions['fraud_flag'].mean()
    print(f"Proportion of fraudulent transfers involving these dual-role accounts: {fraud_proportion_dual_role:.4f}")
else:
    print("No transactions found for accounts that are both fraudulent origin and destination.")

In [ ]:
# Comptes distincts du jeu de données
all_unique_accounts = pd.concat([df['origin_account'], df['destination_account']]).unique()

# Comptes ayant déjà fraudé
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Comptes jamais frauduleux : dans all_unique_accounts, hors all_fraudulent_accounts
truly_non_fraudulent_accounts = set(all_unique_accounts) - all_fraudulent_accounts
print(f"Total number of truly non-fraudulent accounts (never involved in fraud): {len(truly_non_fraudulent_accounts)}")

# Ces comptes jamais frauduleux jouent-ils les deux rôles ?
non_fraudulent_origin_only = df[df['origin_account'].isin(truly_non_fraudulent_accounts)]['origin_account'].unique()
non_fraudulent_destination_only = df[df['destination_account'].isin(truly_non_fraudulent_accounts)]['destination_account'].unique()

non_fraudulent_origin_and_destination = set(non_fraudulent_origin_only).intersection(set(non_fraudulent_destination_only))
print(f"Number of truly non-fraudulent accounts that act as both origin and destination: {len(non_fraudulent_origin_and_destination)}")

# Transactions de ces comptes non frauduleux à double rôle
df_non_fraud_dual_role_transactions = df[
    (df['origin_account'].isin(non_fraudulent_origin_and_destination)) |
    (df['destination_account'].isin(non_fraudulent_origin_and_destination))
].copy()

print(f"\nTotal transactions involving accounts that are truly non-fraudulent and act as both origin and destination: {len(df_non_fraud_dual_role_transactions)}")

# Vérification qu'ils envoient et reçoivent
sends_money_nf = df_non_fraud_dual_role_transactions['origin_account'].isin(non_fraudulent_origin_and_destination).any()
receives_money_nf = df_non_fraud_dual_role_transactions['destination_account'].isin(non_fraudulent_origin_and_destination).any()
print(f"Do these dual-role non-fraudulent accounts send money in these transactions? {sends_money_nf}")
print(f"Do these dual-role non-fraudulent accounts receive money in these transactions? {receives_money_nf}")

# Part de transferts frauduleux parmi ceux de ces comptes
# Par définition, un compte jamais frauduleux n'est associé à aucun fraud_flag = 1
if not df_non_fraud_dual_role_transactions.empty:
    fraud_proportion_non_fraud_dual_role = df_non_fraud_dual_role_transactions['fraud_flag'].mean()
    print(f"Proportion of fraudulent transfers involving these truly non-fraudulent dual-role accounts: {fraud_proportion_non_fraud_dual_role:.4f}")
else:
    print("No transactions found for these truly non-fraudulent dual-role accounts.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Nombre de fraudes par période
fraud_by_period = df.groupby('period')['fraud_flag'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='period', y='fraud_flag', data=fraud_by_period)
plt.title('Evolution du nombre de fraudes par période')
plt.xlabel('Période')
plt.ylabel('Nombre de fraudes')
plt.grid(True)
plt.show()

### Corrélation entre comptes classés et fraude

Corrélation de Pearson entre `origin_account_ranked`, `destination_account_ranked` et `fraud_flag`, pour tester une relation linéaire.

### Taux de fraude moyen par période

Taux de fraude période par période : une variation forte signalerait des moments où la détection a été contournée, ou un changement de tactique des fraudeurs.

In [ ]:
# Taux de fraude moyen par période
fraud_rate_by_period = df.groupby('period')['fraud_flag'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='period', y='fraud_flag', data=fraud_rate_by_period)
plt.title('Taux de Fraude Moyen par Période')
plt.xlabel('Période')
plt.ylabel('Taux de Fraude Moyen')
plt.grid(True)
plt.show()

In [ ]:
correlation_ranked_accounts = df[['origin_account_ranked', 'destination_account_ranked', 'fraud_flag']].corr()
print("Matrice de corrélation entre les comptes classés et la fraude :")
display(correlation_ranked_accounts)

### Visualisation de la distribution des comptes classés par statut de fraude

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='origin_account_ranked', hue='fraud_flag', multiple='stack', bins=50, kde=True)
plt.title('Distribution des comptes émetteurs classés par statut de fraude')
plt.xlabel('Rang du compte émetteur')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='destination_account_ranked', hue='fraud_flag', multiple='stack', bins=50, kde=True)
plt.title('Distribution des comptes destinataires classés par statut de fraude')
plt.xlabel('Rang du compte destinataire')
plt.ylabel('Nombre de transactions')
plt.show()

# soumission

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix, average_precision_score
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Rechargement du train complet et restriction à op_03 ---
print("1. Reloading original training data and filtering for 'op_03' operations...")
df = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')
df = df[df['operation'] == 'op_03'].copy() # op_03 uniquement

# Tri par période pour les features séquentielles
df = df.sort_values(by='period').reset_index(drop=True)

# --- 2. Pipeline de features ---
print("2. Applying feature engineering to 'op_03' training data...")

# 2.1 Features séquentielles
df['origin_transaction_sequence'] = df.groupby('origin_account').cumcount() + 1
df['destination_transaction_sequence'] = df.groupby('destination_account').cumcount() + 1
df['origin_dest_pair_sequence'] = df.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2.2 Drapeaux d'historique de fraude
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# Les autres étapes (classement des comptes, hexadécimal, décimales, motifs de fraude)
# se sont révélées moins informatives : on garde les features les plus importantes.

print("Feature engineering on 'op_03' training data complete.")

# --- 3. Sous-modèle (model_ever_fraud_dest) ---
print("3. Training sub-model to predict 'destination_account_previously_fraud' on 'op_03' data...")

Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df[Y_EVER_FRAUD_TARGET]

# Features du sous-modèle, réduites aux plus informatives
features_for_ever_fraud_prediction_model = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence'
]
features_for_ever_fraud_prediction_model = [f for f in features_for_ever_fraud_prediction_model if f in df.columns]
X_ever_fraud = df[features_for_ever_fraud_prediction_model]
X_ever_fraud = X_ever_fraud.fillna(0) # NaN remplis à 0

X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
    X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
)
neg_count_ef = y_train_ef.value_counts()[0]
pos_count_ef = y_train_ef.value_counts()[1]
scale_pos_weight_ef = neg_count_ef / pos_count_ef

model_ever_fraud_dest = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_ef
)
model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

# predicted_dest_ever_fraud_proba sur tout df
df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_ever_fraud)[:, 1]

print("Sub-model training complete and 'predicted_dest_ever_fraud_proba' added to training data.")

# Évaluation du sous-modèle et matrice de confusion
y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
precision_ef = precision_score(y_test_ef, y_pred_ef)
recall_ef = recall_score(y_test_ef, y_pred_ef)
roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

print(f"\n--- Sub-Model Performance (predicting destination_account_previously_fraud) ---")
print(f"Accuracy: {accuracy_ef:.4f}")
print(f"Precision: {precision_ef:.4f}")
print(f"Recall: {recall_ef:.4f}")
print(f"ROC AUC: {roc_auc_ef:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
            yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
plt.title('Confusion Matrix for Sub-Model (Ever Fraudulent Destination Account)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# --- 4. Modèle principal de détection de fraude (model_main_fraud) ---
print("4. Training main fraud detection model on 'op_03' data...")

# Censure du df pour l'entraînement du modèle principal

all_unique_accounts_set = set(pd.concat([df['origin_account'], df['destination_account']]).unique())
all_fraudulent_accounts_set = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
truly_non_fraudulent_accounts_set = all_unique_accounts_set - all_fraudulent_accounts_set

first_appearance_all = df.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})
account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_set)].copy()

# mean_time_to_fraud_dest vaut 2.1948, d'où long_time_threshold_periods = 6
long_time_threshold_periods = 6 # 2.1948 * 3, arrondi au supérieur

stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods]['destination_account'].tolist())

df_filtered_for_training = df[
    (df['fraud_flag'] == 1) |
    ((df['fraud_flag'] == 0) & (df['destination_account'].isin(stable_long_time_accounts_set)))
].copy()

y_main = df_filtered_for_training['fraud_flag']

# Features du modèle principal, dont la prédiction du sous-modèle
main_model_features = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence',
    'predicted_dest_ever_fraud_proba' # This is the crucial feature from the sub-model
]
main_model_features = [f for f in main_model_features if f in df_filtered_for_training.columns]
X_main = df_filtered_for_training[main_model_features]
X_main = X_main.fillna(0) # NaN remplis à 0

X_train_main, X_test_main, y_train_main, y_test_main = train_test_split(
    X_main, y_main, test_size=0.3, random_state=42, stratify=y_main
)
neg_count_main = y_train_main.value_counts()[0]
pos_count_main = y_train_main.value_counts()[1]
scale_pos_weight_main = neg_count_main / pos_count_main

model_main_fraud = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_main
)
model_main_fraud.fit(X_train_main, y_train_main)

print("Main model training complete.")

# Évaluation du modèle principal et matrice de confusion
y_pred_main = model_main_fraud.predict(X_test_main)
y_pred_proba_main = model_main_fraud.predict_proba(X_test_main)[:, 1]

accuracy_main = accuracy_score(y_test_main, y_pred_main)
precision_main = precision_score(y_test_main, y_pred_main)
recall_main = recall_score(y_test_main, y_pred_main)
roc_auc_main = roc_auc_score(y_test_main, y_pred_proba_main)
average_precision_main = average_precision_score(y_test_main, y_pred_proba_main)

print(f"\n--- Main Model Performance (predicting fraud_flag) ---")
print(f"Accuracy: {accuracy_main:.4f}")
print(f"Precision: {precision_main:.4f}")
print(f"Recall: {recall_main:.4f}")
print(f"ROC AUC: {roc_auc_main:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_main:.4f}")

cm_main = confusion_matrix(y_test_main, y_pred_main)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_main, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Main Model (Fraud Flag)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# --- 5. Traitement du test et prédictions ---
print("5. Processing test data and generating predictions...")

test_df = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')
test_df = test_df[test_df['operation'] == 'op_03'].copy() # test restreint à op_03
submission_df = test_df[['id']].copy()

# Mêmes features appliquées à test_df
# Tri de test_df par période
test_df = test_df.sort_values(by='period').reset_index(drop=True)

# Features séquentielles
test_df['origin_transaction_sequence'] = test_df.groupby('origin_account').cumcount() + 1
test_df['destination_transaction_sequence'] = test_df.groupby('destination_account').cumcount() + 1
test_df['origin_dest_pair_sequence'] = test_df.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Drapeaux d'historique de fraude, issus du train
test_df['origin_account_previously_fraud'] = test_df['origin_account'].isin(all_fraudulent_accounts).astype(int)
test_df['destination_account_previously_fraud'] = test_df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# Les étapes écartées (classement des comptes, hexadécimal, décimales, motifs de fraude)
# ne sont pas générées pour test_df : ni origin_account_ranked,
# ni destination_account_ranked, ni calc_origin_decimal, ni les features de motif.

# predicted_dest_ever_fraud_proba sur test_df
X_test_ever_fraud_predict = test_df[features_for_ever_fraud_prediction_model].fillna(0)
test_df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_test_ever_fraud_predict)[:, 1]

# Features de la prédiction du modèle principal
X_test_main_predict = test_df[main_model_features].fillna(0)

# Probabilités de fraude finales
submission_df['target'] = model_main_fraud.predict_proba(X_test_main_predict)[:, 1]

# --- 6. Écriture de la soumission ---
submission_df.to_csv('submission.csv', index=False)

print("Full pipeline executed on 'op_03' data. Submission file 'submission.csv' generated.")
print("First 5 rows of submission.csv:")
print(submission_df.head())


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix, average_precision_score
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Rechargement du train complet et restriction à op_03 ---
print("1. Reloading original training data and filtering for 'op_03' operations for model training...")
df_train_full = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')
df_op03_train = df_train_full[df_train_full['operation'] == 'op_03'].copy()

# Tri de df_op03_train par période pour les features séquentielles
df_op03_train = df_op03_train.sort_values(by='period').reset_index(drop=True)

# --- 2. Pipeline de features sur le train op_03 ---
print("2. Applying feature engineering to 'op_03' training data...")

# 2.1 Features séquentielles
df_op03_train['origin_transaction_sequence'] = df_op03_train.groupby('origin_account').cumcount() + 1
df_op03_train['destination_transaction_sequence'] = df_op03_train.groupby('destination_account').cumcount() + 1
df_op03_train['origin_dest_pair_sequence'] = df_op03_train.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2.2 Drapeaux d'historique de fraude
# Calculés sur le train complet pour refléter le vrai statut de fraude
fraudulent_origin_accounts_full = df_train_full[df_train_full['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts_full = df_train_full[df_train_full['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts_full = set(fraudulent_origin_accounts_full).union(set(fraudulent_destination_accounts_full))
df_op03_train['origin_account_previously_fraud'] = df_op03_train['origin_account'].isin(all_fraudulent_accounts_full).astype(int)
df_op03_train['destination_account_previously_fraud'] = df_op03_train['destination_account'].isin(all_fraudulent_accounts_full).astype(int)

print("Feature engineering on 'op_03' training data complete.")

# --- 3. Sous-modèle (model_ever_fraud_dest) ---
print("3. Training sub-model to predict 'destination_account_previously_fraud' on 'op_03' training data...")

Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df_op03_train[Y_EVER_FRAUD_TARGET]

# Features du sous-modèle, réduites aux plus informatives
features_for_ever_fraud_prediction_model = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence'
]
features_for_ever_fraud_prediction_model = [f for f in features_for_ever_fraud_prediction_model if f in df_op03_train.columns]
X_ever_fraud = df_op03_train[features_for_ever_fraud_prediction_model]
X_ever_fraud = X_ever_fraud.fillna(0) # NaN remplis à 0

X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
    X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
)
neg_count_ef = y_train_ef.value_counts()[0]
pos_count_ef = y_train_ef.value_counts()[1]
scale_pos_weight_ef = neg_count_ef / pos_count_ef

model_ever_fraud_dest = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_ef
)
model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

# predicted_dest_ever_fraud_proba sur tout df_op03_train
df_op03_train['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_ever_fraud)[:, 1]

print("Sub-model training complete and 'predicted_dest_ever_fraud_proba' added to training data.")

# Évaluation du sous-modèle et matrice de confusion
y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
precision_ef = precision_score(y_test_ef, y_pred_ef)
recall_ef = recall_score(y_test_ef, y_pred_ef)
roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

print(f"\n--- Sub-Model Performance (predicting destination_account_previously_fraud) ---")
print(f"Accuracy: {accuracy_ef:.4f}")
print(f"Precision: {precision_ef:.4f}")
print(f"Recall: {recall_ef:.4f}")
print(f"ROC AUC: {roc_auc_ef:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
            yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
plt.title('Confusion Matrix for Sub-Model (Ever Fraudulent Destination Account)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# --- 4. Modèle principal de détection de fraude (model_main_fraud) ---
print("4. Training main fraud detection model on 'op_03' training data...")

# Censure de df_op03_train pour le modèle principal
# all_unique_accounts_full dérivé du train complet, pour la généralisation
all_unique_accounts_full_set = set(pd.concat([df_train_full['origin_account'], df_train_full['destination_account']]).unique())
truly_non_fraudulent_accounts_full_set = all_unique_accounts_full_set - all_fraudulent_accounts_full

first_appearance_all = df_train_full.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df_train_full.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})
account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_full_set)].copy()

long_time_threshold_periods = 6 # mean_time_to_fraud_dest * 3, arrondi au supérieur

stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods]['destination_account'].tolist())

df_filtered_for_training_main = df_op03_train[
    (df_op03_train['fraud_flag'] == 1) |
    ((df_op03_train['fraud_flag'] == 0) & (df_op03_train['destination_account'].isin(stable_long_time_accounts_set)))
].copy()

y_main = df_filtered_for_training_main['fraud_flag']

# Features du modèle principal, dont la prédiction du sous-modèle
main_model_features = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence',
    'predicted_dest_ever_fraud_proba' # This is the crucial feature from the sub-model
]
main_model_features = [f for f in main_model_features if f in df_filtered_for_training_main.columns]
X_main = df_filtered_for_training_main[main_model_features]
X_main = X_main.fillna(0) # NaN remplis à 0

X_train_main, X_test_main, y_train_main, y_test_main = train_test_split(
    X_main, y_main, test_size=0.3, random_state=42, stratify=y_main
)
neg_count_main = y_train_main.value_counts()[0]
pos_count_main = y_train_main.value_counts()[1]
scale_pos_weight_main = neg_count_main / pos_count_main

model_main_fraud = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_main
)
model_main_fraud.fit(X_train_main, y_train_main)

print("Main model training complete.")

# Évaluation du modèle principal et matrice de confusion
y_pred_main = model_main_fraud.predict(X_test_main)
y_pred_proba_main = model_main_fraud.predict_proba(X_test_main)[:, 1]

accuracy_main = accuracy_score(y_test_main, y_pred_main)
precision_main = precision_score(y_test_main, y_pred_main)
recall_main = recall_score(y_test_main, y_pred_main)
roc_auc_main = roc_auc_score(y_test_main, y_pred_proba_main)
average_precision_main = average_precision_score(y_test_main, y_pred_proba_main)

print(f"\n--- Main Model Performance (predicting fraud_flag) ---")
print(f"Accuracy: {accuracy_main:.4f}")
print(f"Precision: {precision_main:.4f}")
print(f"Recall: {recall_main:.4f}")
print(f"ROC AUC: {roc_auc_main:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_main:.4f}")

cm_main = confusion_matrix(y_test_main, y_pred_main)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_main, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Main Model (Fraud Flag)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# --- 5. Traitement du test et prédictions pour submission2.csv ---
print("5. Processing test data and generating predictions for 'submission2.csv'...")

test_df_full = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')
submission_df_2 = test_df_full[['id']].copy()

# Cible initialisée à 0.0 pour les transactions hors op_03
submission_df_2['target'] = 0.0

# Transactions op_03 isolées pour le pipeline complet
test_df_op03 = test_df_full[test_df_full['operation'] == 'op_03'].copy()

# Mêmes features appliquées à test_df_op03
test_df_op03 = test_df_op03.sort_values(by='period').reset_index(drop=True)

# Features séquentielles
test_df_op03['origin_transaction_sequence'] = test_df_op03.groupby('origin_account').cumcount() + 1
test_df_op03['destination_transaction_sequence'] = test_df_op03.groupby('destination_account').cumcount() + 1
test_df_op03['origin_dest_pair_sequence'] = test_df_op03.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Drapeaux d'historique de fraude, issus de all_fraudulent_accounts_full
test_df_op03['origin_account_previously_fraud'] = test_df_op03['origin_account'].isin(all_fraudulent_accounts_full).astype(int)
test_df_op03['destination_account_previously_fraud'] = test_df_op03['destination_account'].isin(all_fraudulent_accounts_full).astype(int)

# predicted_dest_ever_fraud_proba sur test_df_op03
X_test_ever_fraud_predict_op03 = test_df_op03[features_for_ever_fraud_prediction_model].fillna(0)
test_df_op03['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_test_ever_fraud_predict_op03)[:, 1]

# Features de la prédiction du modèle principal
X_test_main_predict_op03 = test_df_op03[main_model_features].fillna(0)

# Probabilités de fraude finales sur les transactions op_03
final_probabilities_op03 = model_main_fraud.predict_proba(X_test_main_predict_op03)[:, 1]

# Mise à jour de target dans submission_df_2 pour les transactions op_03
submission_df_2.loc[submission_df_2['id'].isin(test_df_op03['id']), 'target'] = final_probabilities_op03

# --- 6. Écriture de la soumission ---
submission_df_2.to_csv('submission2.csv', index=False)

print("Full pipeline executed with custom non-op_03 handling. Submission file 'submission2.csv' generated.")
print("First 5 rows of submission2.csv:")
print(submission_df_2.head())